# NLP-04 — 11_final_documentation

Publication copy of the research notebook.

Saved outputs and unnecessary metadata have been removed. Private annotations, raw datasets and model checkpoints are retained separately.

Some cells require external research artifacts. The public notebook is not a standalone reproduction package.

In [ ]:
# Cell 1: Final research documentation setup

from google.colab import drive
from pathlib import Path

import hashlib
import json
import gc
import numpy as np
import pandas as pd

from IPython.display import display

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir()

CONFIG_DIR = PROJECT_ROOT / '00_config'
DATA_DIR = PROJECT_ROOT / '01_data'
NOTEBOOKS_DIR = PROJECT_ROOT / '02_notebooks'
SRC_DIR = PROJECT_ROOT / '03_src'
MODELS_DIR = PROJECT_ROOT / '04_models'

NOTEBOOK_DIR = NOTEBOOKS_DIR / '11_final_documentation'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
LOGS_DIR = RESULTS_DIR / 'logs'
REPORTS_DIR = RESULTS_DIR / 'reports'
ASSETS_DIR = RESULTS_DIR / 'selected_assets'

for path in [
    TABLES_DIR,
    RUN_CONFIGS_DIR,
    LOGS_DIR,
    REPORTS_DIR,
    ASSETS_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42


# File utility functions

def read_json(path):

    return json.loads(
        Path(path).read_text(encoding='utf-8')
    )


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024), b''
        ):
            digest.update(chunk)

    return digest.hexdigest()


def save_json_atomic(path, content):

    path = Path(path)

    temp = path.with_suffix(path.suffix + '.tmp')

    temp.write_text(
        json.dumps(
            content,
            indent=4,
            allow_nan=False
        ),
        encoding='utf-8'
    )

    assert read_json(temp) == content

    temp.replace(path)


def save_csv_atomic(path, frame):

    path = Path(path)

    temp = path.with_suffix(path.suffix + '.tmp')

    frame.to_csv(temp, index=False)

    temp.replace(path)


def verify_file(record):

    path = (PROJECT_ROOT / record['path']).resolve()

    assert path.is_relative_to(PROJECT_ROOT.resolve())

    assert path.is_file(), path

    assert path.stat().st_size == record['size_bytes']

    assert calculate_sha256(path) == record['sha256'], path

    return path


# Part A: Verify frozen research protocol

FROZEN_MANIFEST_PATH = (
    NOTEBOOKS_DIR /
    '06_model_optimization/results/configs/frozen_manifest.json'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

EXPECTED_FROZEN_HASH = (
    '229adb5d6166a626f6b7f7fddecf38b4'
    'de58d21e56ec4367465ad29174ef2fd1'
)

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

frozen_manifest = read_json(FROZEN_MANIFEST_PATH)

assert frozen_manifest['selection_partition'] == 'validation'

assert frozen_manifest['thresholds_frozen']

assert frozen_manifest['model_parameters_frozen']

assert frozen_manifest['final_project_model_frozen']

assert not frozen_manifest['official_test_used_for_selection']

assert not frozen_manifest['post_freeze_refitting_permitted']

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

LABELS = frozen_manifest['labels']

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

FROZEN_MODELS = {
    entry['run_id']: entry
    for entry in frozen_manifest['models']
}

assert len(FROZEN_MODELS) == 19

PRIMARY_RUN_ID = frozen_manifest[
    'primary_selected_run_id'
]

assert PRIMARY_RUN_ID == (
    'S02_ROBERTA_FOCAL_FULL_S42'
)

primary_model_entry = FROZEN_MODELS[PRIMARY_RUN_ID]

registry = pd.read_csv(REGISTRY_PATH)

decision_log = pd.read_csv(DECISION_LOG_PATH)

print('Frozen research protocol: Verified')
print('Frozen model configurations:', len(FROZEN_MODELS))
print('Selected final model:', PRIMARY_RUN_ID)


# Part B: Restore previous research stages

EVALUATION_RESULTS_DIR = (
    NOTEBOOKS_DIR / '07_final_evaluation/results'
)

ERROR_RESULTS_DIR = (
    NOTEBOOKS_DIR / '08_error_analysis/results'
)

FAIRNESS_RESULTS_DIR = (
    NOTEBOOKS_DIR / '09_fairness_audit/results'
)

EXPLAINABILITY_RESULTS_DIR = (
    NOTEBOOKS_DIR / '10_explainability/results'
)

completion_paths = {

    'Final evaluation': (
        EVALUATION_RESULTS_DIR /
        'logs/evaluation_completion_summary.json'
    ),

    'Error analysis': (
        ERROR_RESULTS_DIR /
        'logs/error_analysis_completion_summary.json'
    ),

    'Fairness audit': (
        FAIRNESS_RESULTS_DIR /
        'logs/fairness_audit_completion_summary.json'
    ),

    'Explainability': (
        EXPLAINABILITY_RESULTS_DIR /
        'logs/explainability_completion_summary.json'
    )
}

completion_records = {
    stage: read_json(path)
    for stage, path in completion_paths.items()
}

evaluation_completion = completion_records[
    'Final evaluation'
]

error_completion = completion_records[
    'Error analysis'
]

fairness_completion = completion_records[
    'Fairness audit'
]

explainability_completion = completion_records[
    'Explainability'
]

assert evaluation_completion['status'] == 'Completed'

assert evaluation_completion[
    'evaluated_configurations'
] == 19

assert error_completion[
    'ai_assisted_review_completed'
]

assert error_completion['review_cases'] == 180

assert fairness_completion['status'] == 'Completed'

assert fairness_completion['identity_terms'] == 30

assert fairness_completion['synthetic_comments'] == 744

assert explainability_completion['status'] == 'Completed'

assert explainability_completion['explained_cases'] == 20

assert explainability_completion[
    'written_interpretations_completed'
]

assert explainability_completion[
    'target_labels_covered'
] == 6

for stage, record in completion_records.items():

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    ), stage

    assert not record['model_training_performed'], stage

    assert not record['thresholds_changed'], stage

    assert not record['test_driven_selection_performed'], stage

    print(stage, ': Completion record verified')


# Part C: Verify previous artifact inventories

expected_inventory_sizes = {
    'Final evaluation': 79,
    'Fairness audit': 29,
    'Explainability': 107
}

verified_inventory = {}
stage_summary_rows = []

for stage, record in completion_records.items():

    inventory = record.get('artifact_inventory', [])

    if stage in expected_inventory_sizes:

        assert len(inventory) == (
            expected_inventory_sizes[stage]
        )

    for artifact in inventory:

        verified_path = verify_file(artifact)

        key = str(
            verified_path.relative_to(PROJECT_ROOT)
        )

        if key in verified_inventory:

            assert verified_inventory[key] == artifact

        else:

            verified_inventory[key] = artifact

    stage_summary_rows.append({

        'Research Stage': stage,

        'Recorded Status': record['status'],

        'Completion Record Path': str(
            completion_paths[stage].relative_to(PROJECT_ROOT)
        ),

        'Completion Record SHA256': calculate_sha256(
            completion_paths[stage]
        ),

        'Inventory Records Verified': len(inventory),

        'Verification Scope': (
            'Completion record and inventoried file hashes'
            if inventory else
            'Completion record only; no inventory supplied'
        ),

        'Frozen Protocol Unchanged': True
    })

    print(
        stage,
        '| Inventory artifacts verified:',
        len(inventory),
        flush=True
    )

stage_summary = pd.DataFrame(
    stage_summary_rows
)

assert len(stage_summary) == 4

save_csv_atomic(
    TABLES_DIR / 'research_stage_verification.csv',
    stage_summary
)


# Part D: Verify official evaluation dataset

EXPECTED_TEST_HASH = (
    'cddc70d876b48b9fe3d5db8109a7404'
    '30a7f4d9f50f9fcd77f4c7d566035f328'
)

TEST_PATH = PROJECT_ROOT / evaluation_completion[
    'test_dataset_path'
]

assert calculate_sha256(
    TEST_PATH
) == EXPECTED_TEST_HASH

assert evaluation_completion[
    'test_dataset_sha256'
] == EXPECTED_TEST_HASH

assert explainability_completion[
    'test_dataset_sha256'
] == EXPECTED_TEST_HASH

test_df = pd.read_parquet(
    TEST_PATH,
    columns=['id', *LABELS]
)

assert len(test_df) == 63978

assert test_df['id'].is_unique

y_test = test_df[LABELS].to_numpy(dtype=np.int8)

assert np.isin(y_test, [0, 1]).all()

print('Official labelled test dataset: Verified')


# Part E: Verify final model dependencies

MODEL_PATH = PROJECT_ROOT / primary_model_entry[
    'model_path'
]

assert MODEL_PATH.is_dir()

verify_file(primary_model_entry['config'])

verify_file(primary_model_entry['threshold_file'])

verify_file(
    primary_model_entry['checkpoint_manifest']
)

for record in primary_model_entry['model_files']:

    verify_file(record)

thresholds = np.array([
    primary_model_entry['optimized_thresholds'][label]
    for label in LABELS
])

assert len(thresholds) == 6

assert np.isfinite(thresholds).all()

assert (
    (thresholds >= 0) &
    (thresholds <= 1)
).all()

print('Final model and tokenizer files: Verified')
print('Six frozen thresholds: Verified')


# Part F: Record outstanding human review

human_review_status = {

    'error_analysis': {
        'ai_assisted_cases': 180,
        'independent_human_validation':
            'Pending unless separately documented'
    },

    'explainability': {
        'written_interpretations': (
            explainability_completion[
                'written_interpretations'
            ]
        ),

        'independent_human_validation_completed': (
            explainability_completion[
                'independent_human_validation_completed'
            ]
        ),

        'pending_cases': (
            explainability_completion[
                'independent_human_review_pending_cases'
            ]
        )
    },

    'synthetic_fairness': {
        'independent_human_validation_completed': (
            fairness_completion[
                'independent_synthetic_human_validation_completed'
            ]
        )
    }
}


# Part G: Establish final documentation configuration

documentation_setup = {

    'notebook': '11_final_documentation',

    'seed': SEED,

    'primary_run_id': PRIMARY_RUN_ID,

    'labels': LABELS,

    'evaluated_configurations': len(FROZEN_MODELS),

    'official_test_samples': len(test_df),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'test_dataset_sha256': EXPECTED_TEST_HASH,

    'verified_previous_artifacts': len(
        verified_inventory
    ),

    'stage_completion_hashes': {
        stage: calculate_sha256(path)
        for stage, path in completion_paths.items()
    },

    'human_review_status': human_review_status,

    'documentation_rules': [

        'Derive reported numerical results from saved artifacts.',

        'Keep validation and official test results separate.',

        'Identify model configurations by experiment run ID.',

        'Distinguish full and representative training subsets.',

        'Document skipped experiments and research limitations.',

        'Do not claim independent human validation where pending.',

        'Do not claim measured deployment latency without evidence.',

        'Exclude private raw comments from public publication.',

        'Use the frozen model as moderation decision support.'
    ],

    'model_training_performed': False,

    'additional_inference_performed': False,

    'thresholds_changed': False,

    'test_driven_selection_performed': False
}

setup_path = (
    RUN_CONFIGS_DIR / 'final_documentation_setup.json'
)

if setup_path.exists():

    assert read_json(setup_path) == documentation_setup

else:

    save_json_atomic(
        setup_path,
        documentation_setup
    )


# Part H: Display verified research headline metrics

headline_results = pd.DataFrame([

    {
        'Metric': 'Official test Macro Average Precision',

        'Value': evaluation_completion[
            'primary_test_macro_ap'
        ]
    },

    {
        'Metric': 'Official test Macro ROC-AUC',

        'Value': evaluation_completion[
            'primary_test_macro_roc_auc'
        ]
    },

    {
        'Metric': 'Official test optimized Macro F1',

        'Value': evaluation_completion[
            'primary_test_optimized_macro_f1'
        ]
    },

    {
        'Metric': 'Official test optimized Micro F1',

        'Value': evaluation_completion[
            'primary_test_optimized_micro_f1'
        ]
    },

    {
        'Metric': 'Explained predictions',

        'Value': explainability_completion[
            'explained_cases'
        ]
    },

    {
        'Metric': 'Mean LIME surrogate R2',

        'Value': explainability_completion[
            'mean_local_surrogate_r2'
        ]
    },

    {
        'Metric': 'Mean absolute sufficiency error',

        'Value': explainability_completion[
            'mean_absolute_sufficiency_error'
        ]
    }

])

save_csv_atomic(
    TABLES_DIR / 'verified_headline_results.csv',
    headline_results
)

print('\nResearch stage verification')

display(stage_summary.drop(
    columns=[
        'Completion Record Path',
        'Completion Record SHA256'
    ]
))

print('\nVerified headline research results')

display(headline_results.round(6))

print('\nFrozen decision thresholds')

display(pd.DataFrame({
    'Label': LABELS,
    'Frozen Threshold': thresholds
}).round(6))


# Final frozen protocol checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

gc.collect()

print('\nNotebook 11 setup summary')

print('Research completion records verified:', 4)

print(
    'Unique previous inventory artifacts verified:',
    len(verified_inventory)
)

print('Final model:', PRIMARY_RUN_ID)

print('Evaluated model configurations:', len(FROZEN_MODELS))

print('Official test samples:', len(test_df))

print('Written LIME interpretations:', 20)

print('Independent human validation: Pending')

print('Model training and inference: None')

print('Frozen protocol: Unchanged')

print('Cell 1 completed successfully.')

In [ ]:
# Cell 2: Consolidate final experimental results

# Load only previously verified evaluation artifacts

evaluation_inventory = {
    record['path']: record
    for record in evaluation_completion['artifact_inventory']
}


def load_evaluation_table(filename):

    path = EVALUATION_RESULTS_DIR / 'tables' / filename

    key = str(path.relative_to(PROJECT_ROOT))

    verify_file(evaluation_inventory[key])

    return pd.read_csv(
        path,
        float_precision='round_trip'
    )


comparison = load_evaluation_table(
    'test_model_comparison.csv'
)

aggregate = load_evaluation_table(
    'test_aggregate_metrics.csv'
)

per_label = load_evaluation_table(
    'test_per_label_metrics.csv'
)

runtime = load_evaluation_table(
    'test_runtime_model_size.csv'
)

confidence_intervals = load_evaluation_table(
    'test_bootstrap_intervals.csv'
)

paired_intervals = load_evaluation_table(
    'test_paired_bootstrap_intervals.csv'
)

matched_comparisons = load_evaluation_table(
    'test_matched_research_comparisons.csv'
)

threshold_effect = load_evaluation_table(
    'test_threshold_effect.csv'
)

prediction_index = load_evaluation_table(
    'test_prediction_index.csv'
)

assert len(comparison) == 19
assert len(aggregate) == 38
assert len(per_label) == 228
assert len(runtime) == 19
assert len(confidence_intervals) == 84
assert len(paired_intervals) == 9
assert len(prediction_index) == 19

assert set(comparison['Run ID']) == set(FROZEN_MODELS)

assert comparison['Run ID'].is_unique
assert runtime['Run ID'].is_unique

assert aggregate['Test Samples'].eq(63978).all()

print('Original evaluation tables: Verified')


# Part A: Consolidate validation results separately

registry_index = registry.set_index('run_id')

assert registry_index.index.is_unique

validation_rows = []

for rid, entry in FROZEN_MODELS.items():

    reg = registry_index.loc[rid]

    assert reg['status'] == 'Completed'

    assert int(reg['training_samples']) == (
        entry['training_samples']
    )

    np.testing.assert_allclose(
        float(reg['validation_macro_ap']),
        entry['validation_macro_ap'],
        atol=1e-10,
        rtol=0
    )

    validation_rows.append({

        'Run ID': rid,

        'Model': entry['model'],
        'Kind': entry['kind'],

        'Feature Representation': (
            entry['feature_representation']
        ),

        'Loss Function': reg['loss_function'],

        'Sampling Strategy': entry['sampling_strategy'],
        'Training Samples': entry['training_samples'],

        'Seed': entry['seed'],

        'Validation Macro AP': (
            entry['validation_macro_ap']
        ),

        'Validation Optimized Macro F1': (
            entry['validation_optimized_macro_f1']
        ),

        'Training Seconds': (
            entry['training_time_seconds']
        ),

        'Primary Selected Model': (
            rid == PRIMARY_RUN_ID
        ),

        'Selection Partition': 'Validation'
    })

validation_results = pd.DataFrame(
    validation_rows
)

rank_lookup = comparison.set_index(
    'Run ID'
)['Validation Rank']

validation_results['Validation Rank'] = (
    validation_results['Run ID'].map(rank_lookup)
)

validation_results = (
    validation_results
    .sort_values('Validation Rank')
    .reset_index(drop=True)
)

assert validation_results.iloc[0]['Run ID'] == (
    PRIMARY_RUN_ID
)

assert validation_results[
    'Validation Macro AP'
].is_monotonic_decreasing

save_csv_atomic(
    TABLES_DIR / 'final_validation_results.csv',
    validation_results
)

print('Validation comparison: 19 configurations')


# Part B: Consolidate official test results

test_results = aggregate.merge(

    validation_results[[
        'Run ID',
        'Sampling Strategy',
        'Feature Representation',
        'Loss Function',
        'Seed'
    ]],

    on='Run ID',
    validate='many_to_one'
)

test_results['Evaluation Partition'] = (
    'Official labelled test'
)

assert len(test_results) == 38

assert not test_results.duplicated([
    'Run ID',
    'Operating Point'
]).any()

save_csv_atomic(
    TABLES_DIR / 'final_official_test_results.csv',
    test_results
)


# Final comparison index:
# Validation-based ranking, with explicitly named test metrics

test_optimized = test_results[
    test_results['Operating Point'] == 'optimized'
].copy()

assert len(test_optimized) == 19

test_columns = [

    'Run ID',

    'Macro AP',
    'Macro ROC-AUC',

    'Macro Precision',
    'Macro Recall',

    'Macro F1',
    'Micro F1',

    'Hamming Loss',
    'Exact Match Accuracy',

    'Flagged Comments',
    'Flagged Per 1000 Comments'
]

test_for_comparison = (
    test_optimized[test_columns].rename(
        columns={
            col: f'Test {col}'
            for col in test_columns
            if col != 'Run ID'
        }
    )
)

final_comparison = validation_results.merge(

    test_for_comparison,

    on='Run ID',
    validate='one_to_one'
)

final_comparison = final_comparison.merge(

    runtime[[
        'Run ID',
        'Inference Seconds',
        'Inference Device',
        'Model Size MiB',
        'Comments Per Second (Inference Only)',
        'Peak GPU Allocated GiB',
        'Peak GPU Reserved GiB',
        'Batch Size'
    ]],

    on='Run ID',
    validate='one_to_one'
)

final_comparison['Test Operating Point'] = (
    'Frozen validation-optimized thresholds'
)

assert len(final_comparison) == 19

primary_result = final_comparison[
    final_comparison['Run ID'] == PRIMARY_RUN_ID
].iloc[0]

np.testing.assert_allclose(

    [
        primary_result['Test Macro AP'],
        primary_result['Test Macro ROC-AUC'],
        primary_result['Test Macro F1'],
        primary_result['Test Micro F1']
    ],

    [
        evaluation_completion['primary_test_macro_ap'],
        evaluation_completion['primary_test_macro_roc_auc'],
        evaluation_completion['primary_test_optimized_macro_f1'],
        evaluation_completion['primary_test_optimized_micro_f1']
    ],

    atol=1e-12,
    rtol=0
)

save_csv_atomic(
    TABLES_DIR / 'T01_final_comparison_index.csv',
    final_comparison
)

print('Final model comparison index: Verified')


# Part C: Restore final per-label metrics

primary_per_label = per_label[
    (per_label['Run ID'] == PRIMARY_RUN_ID) &
    (per_label['Operating Point'] == 'optimized')
].copy()

primary_per_label = (
    primary_per_label
    .set_index('Label')
    .loc[LABELS]
    .reset_index()
)

assert len(primary_per_label) == 6

np.testing.assert_allclose(

    primary_per_label['Threshold'],

    thresholds,

    atol=1e-12,
    rtol=0
)

assert (
    primary_per_label['TP'] +
    primary_per_label['TN'] +
    primary_per_label['FP'] +
    primary_per_label['FN']
).eq(len(test_df)).all()

assert (
    primary_per_label['TP'] +
    primary_per_label['FN']
).eq(
    primary_per_label['Positive Support']
).all()

save_csv_atomic(
    TABLES_DIR / 'final_primary_per_label_metrics.csv',
    primary_per_label
)


# Part D: Restore frozen final predictions

prediction_record = prediction_index.set_index(
    'Run ID'
).loc[PRIMARY_RUN_ID]

primary_prediction_path = PROJECT_ROOT / (
    prediction_record['Prediction Path']
)

verify_file(
    evaluation_inventory[
        str(primary_prediction_path.relative_to(PROJECT_ROOT))
    ]
)

assert calculate_sha256(
    primary_prediction_path
) == prediction_record['SHA256']

primary_predictions = pd.read_parquet(
    primary_prediction_path
)

assert len(primary_predictions) == len(test_df)
assert primary_predictions['id'].is_unique
assert primary_predictions['run_id'].eq(PRIMARY_RUN_ID).all()
assert primary_predictions['split'].eq('test_official').all()

primary_predictions['id'] = (
    primary_predictions['id'].astype(str)
)

test_ids = test_df['id'].astype(str)

assert set(primary_predictions['id']) == set(test_ids)

primary_predictions = (
    primary_predictions
    .set_index('id')
    .loc[test_ids]
    .reset_index()
)

np.testing.assert_array_equal(

    primary_predictions[
        [f'true_{label}' for label in LABELS]
    ].to_numpy(),

    y_test
)

scores = primary_predictions[
    [f'score_{label}' for label in LABELS]
].to_numpy(dtype=np.float64)

pred_optimized = primary_predictions[
    [f'pred_optimized_{label}' for label in LABELS]
].to_numpy(dtype=np.int8)

pred_default = primary_predictions[
    [f'pred_default_{label}' for label in LABELS]
].to_numpy(dtype=np.int8)

assert np.isfinite(scores).all()
assert ((scores >= 0) & (scores <= 1)).all()

np.testing.assert_array_equal(
    pred_optimized,
    (scores >= thresholds).astype(np.int8)
)

np.testing.assert_array_equal(
    pred_default,
    (
        scores >= primary_model_entry['default_threshold']
    ).astype(np.int8)
)

# Verify per-label confusion counts from saved predictions

for j, label in enumerate(LABELS):

    actual = y_test[:, j]
    pred = pred_optimized[:, j]

    row = primary_per_label.iloc[j]

    assert row['Label'] == label

    assert int(row['TP']) == int(
        ((actual == 1) & (pred == 1)).sum()
    )

    assert int(row['TN']) == int(
        ((actual == 0) & (pred == 0)).sum()
    )

    assert int(row['FP']) == int(
        ((actual == 0) & (pred == 1)).sum()
    )

    assert int(row['FN']) == int(
        ((actual == 1) & (pred == 0)).sum()
    )

print('Final prediction scores and confusion counts: Verified')


# Part E: Thresholds and benchmark moderation workload

n = len(test_df)

threshold_cost = primary_per_label[[
    'Run ID',
    'Label',
    'Threshold',
    'Positive Support',
    'Negative Support',
    'Predicted Positives',
    'TP',
    'TN',
    'FP',
    'FN',
    'Precision',
    'Recall',
    'F1'
]].copy()

threshold_cost['Flags Per 1000 Comments'] = (
    1000 * threshold_cost['Predicted Positives'] / n
)

threshold_cost['False Flags Per 1000 Comments'] = (
    1000 * threshold_cost['FP'] / n
)

threshold_cost['Missed Positives Per 1000 Comments'] = (
    1000 * threshold_cost['FN'] / n
)

threshold_cost['Threshold Selection Partition'] = (
    'Validation'
)

threshold_cost['Cost Evaluation Partition'] = (
    'Official labelled test'
)

save_csv_atomic(
    TABLES_DIR / 'T03_final_thresholds_and_cost.csv',
    threshold_cost
)


# Unique comment-level moderation workload
# Do not sum individual label flags as unique comments

actual_any = y_test.any(axis=1)

workload_rows = []

for operating_point, predictions in [

    ('Default', pred_default),

    ('Validation optimized', pred_optimized)
]:

    flagged_any = predictions.any(axis=1)

    tp = int((actual_any & flagged_any).sum())
    fp = int((~actual_any & flagged_any).sum())

    fn = int((actual_any & ~flagged_any).sum())
    tn = int((~actual_any & ~flagged_any).sum())

    assert tp + fp + fn + tn == n

    total_flags = int(flagged_any.sum())

    workload_rows.append({

        'Run ID': PRIMARY_RUN_ID,

        'Operating Point': operating_point,

        'Benchmark Comments': n,

        'Actual Any Label Positive': int(actual_any.sum()),

        'Flagged Unique Comments': total_flags,

        'Flagged Positive Comments': tp,

        'Flagged All-Label Clean Comments': fp,

        'Missed Positive Comments': fn,

        'Unflagged Clean Comments': tn,

        'Flagged Comments Per 1000': (
            1000 * total_flags / n
        ),

        'False Flagged Comments Per 1000': (
            1000 * fp / n
        ),

        'Missed Positive Comments Per 1000': (
            1000 * fn / n
        ),

        'Comment-Level Precision': (
            tp / (tp + fp)
        ),

        'Comment-Level Recall': (
            tp / (tp + fn)
        ),

        'Comment-Level FPR': (
            fp / (fp + tn)
        ),

        'Interpretation': (
            'Observed benchmark workload; '
            'not a production forecast'
        )
    })

moderation_workload = pd.DataFrame(
    workload_rows
)

optimized_workload = moderation_workload.iloc[1]

assert optimized_workload[
    'Flagged Unique Comments'
] == primary_result['Test Flagged Comments']

np.testing.assert_allclose(

    optimized_workload['Flagged Comments Per 1000'],

    primary_result['Test Flagged Per 1000 Comments'],

    atol=1e-10,
    rtol=0
)

save_csv_atomic(
    TABLES_DIR / 'final_moderation_workload.csv',
    moderation_workload
)

print('Moderation workload: Verified')


# Part F: Preserve uncertainty and matched comparisons

selected_model_ids = set(
    frozen_manifest['selected_models'].values()
)

primary_intervals = confidence_intervals[
    confidence_intervals['Run ID'] == PRIMARY_RUN_ID
].copy()

assert len(primary_intervals) > 0

assert paired_intervals[
    'Candidate Run'
].eq(PRIMARY_RUN_ID).all()

output_tables = {

    'final_primary_confidence_intervals.csv':
        primary_intervals,

    'final_paired_model_comparisons.csv':
        paired_intervals,

    'final_matched_research_comparisons.csv':
        matched_comparisons,

    'final_threshold_effect.csv':
        threshold_effect,

    'final_runtime_and_model_size.csv':
        runtime
}

for filename, frame in output_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        frame
    )


# Part G: Record measured computational efficiency

primary_runtime = runtime[
    runtime['Run ID'] == PRIMARY_RUN_ID
].iloc[0]

throughput = float(
    primary_runtime['Comments Per Second (Inference Only)']
)

assert throughput > 0

efficiency_summary = {

    'run_id': PRIMARY_RUN_ID,

    'benchmark_comments': n,

    'training_seconds': float(
        primary_runtime['Training Seconds']
    ),

    'offline_test_inference_seconds': float(
        primary_runtime['Inference Seconds']
    ),

    'offline_inference_comments_per_second': throughput,

    'amortized_inference_ms_per_comment': (
        1000 / throughput
    ),

    'inference_device': str(
        primary_runtime['Inference Device']
    ),

    'batch_size': int(
        primary_runtime['Batch Size']
    ),

    'model_size_mib': float(
        primary_runtime['Model Size MiB']
    ),

    'peak_gpu_allocated_gib': float(
        primary_runtime['Peak GPU Allocated GiB']
    ),

    'peak_gpu_reserved_gib': float(
        primary_runtime['Peak GPU Reserved GiB']
    ),

    'single_request_latency_measured': False,

    'end_to_end_production_latency_measured': False,

    'production_deployment_tested': False,

    'limitations': [

        'Offline batched inference is not production latency.',

        'Reported transformer inference used cached tokenized inputs.',

        'GPU allocator measurements exclude some system overhead.',

        'Model size is not total application memory usage.'
    ]
}

efficiency_path = (
    RUN_CONFIGS_DIR / 'final_model_efficiency.json'
)

save_json_atomic(
    efficiency_path,
    efficiency_summary
)


# Part H: Verify and inventory consolidated artifacts

result_table_names = [

    'final_validation_results.csv',
    'final_official_test_results.csv',
    'T01_final_comparison_index.csv',
    'final_primary_per_label_metrics.csv',
    'T03_final_thresholds_and_cost.csv',
    'final_moderation_workload.csv',

    *output_tables.keys()
]

consolidation_inventory = []

for path in [

    *[
        TABLES_DIR / name
        for name in result_table_names
    ],

    efficiency_path
]:

    consolidation_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    })

for record in consolidation_inventory:
    verify_file(record)

assert len(result_table_names) == 11
assert len(consolidation_inventory) == 12

# Verify frozen research protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    primary_prediction_path
) == prediction_record['SHA256']


# Record consolidation completion

consolidation_completion = {

    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,

    'validation_configurations': 19,
    'test_configurations': 19,
    'test_operating_point_records': 38,

    'official_test_samples': n,

    'primary_per_label_records': 6,
    'moderation_workload_records': 2,

    'paired_comparison_records': len(paired_intervals),

    'saved_tables': len(result_table_names),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'test_dataset_sha256': EXPECTED_TEST_HASH,

    'primary_prediction_sha256': (
        prediction_record['SHA256']
    ),

    'artifact_inventory': consolidation_inventory,

    'validation_and_test_results_distinguished': True,

    'numerical_results_derived_from_saved_artifacts': True,

    'production_latency_measured': False,

    'model_training_performed': False,

    'additional_inference_performed': False,

    'thresholds_changed': False,

    'test_driven_selection_performed': False
}

save_json_atomic(

    LOGS_DIR / 'final_results_consolidation_summary.json',

    consolidation_completion
)


# Part I: Display final research tables

print('\nFinal comparison: validation-ranked configurations')

display(final_comparison[[

    'Validation Rank',
    'Run ID',
    'Model',
    'Training Samples',

    'Validation Macro AP',
    'Validation Optimized Macro F1',

    'Test Macro AP',
    'Test Macro ROC-AUC',

    'Test Macro F1',
    'Test Micro F1'

]].round(6))


print('\nFinal RoBERTa per-label performance')

display(primary_per_label[[

    'Label',
    'Threshold',
    'Precision',
    'Recall',
    'F1',
    'Average Precision',
    'ROC-AUC',
    'Positive Support'

]].round(6))


print('\nModeration workload per 1000 comments')

display(moderation_workload[[

    'Operating Point',

    'Flagged Unique Comments',

    'Flagged Comments Per 1000',

    'False Flagged Comments Per 1000',

    'Missed Positive Comments Per 1000',

    'Comment-Level Precision',

    'Comment-Level Recall'

]].round(4))


print('\nFinal model computational efficiency')

display(pd.DataFrame([
    efficiency_summary
]).drop(columns='limitations').T)


print('\nCell 2 summary')

print('Validation configurations:', len(validation_results))
print('Official test configurations:', len(test_optimized))
print('Test operating point records:', len(test_results))
print('Final per-label records:', len(primary_per_label))
print('Moderation workload records:', len(moderation_workload))
print('Tables saved:', len(result_table_names))
print('Verified inventory artifacts:', len(consolidation_inventory))
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 2 completed successfully.')

In [ ]:
# Cell 3: Requirement evidence matrix and research limitations

from datetime import datetime, timezone
import pyarrow.parquet as pq

# Restore final consolidation evidence

consolidation_path = (
    LOGS_DIR / 'final_results_consolidation_summary.json'
)

consolidation_completion = read_json(consolidation_path)

assert consolidation_completion['status'] == 'Completed'

assert consolidation_completion[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

for record in consolidation_completion['artifact_inventory']:
    verify_file(record)

print('Consolidated research results: Verified')


# Part A: Verify prepared dataset evidence

dataset_manifest_path = (
    DATA_DIR / 'splits/prepared_dataset_manifest.csv'
)

dataset_manifest = pd.read_csv(dataset_manifest_path)

expected_partitions = {
    'train_full',
    'train_representative',
    'train_enriched',
    'validation',
    'test_official'
}

assert set(dataset_manifest['Partition']) == expected_partitions

dataset_records = dataset_manifest.set_index('Partition')

for row in dataset_manifest.to_dict('records'):

    path = PROJECT_ROOT / row['Dataset Path']

    assert path.stat().st_size == row['Size Bytes']

    assert calculate_sha256(path) == row['SHA256']

    assert pq.ParquetFile(path).metadata.num_rows == row['Rows']

    assert row['Rows'] == row['Unique IDs']

print('Five prepared dataset fingerprints: Verified')

assert dataset_records.loc[
    'validation', 'SHA256'
] == frozen_manifest['validation_dataset_sha256']

assert dataset_records.loc[
    'test_official', 'SHA256'
] == EXPECTED_TEST_HASH

# Verify the dataset associated with each frozen model

training_partitions = {
    dataset_records.loc[p, 'SHA256']: p
    for p in [
        'train_full',
        'train_representative',
        'train_enriched'
    ]
}

for rid, entry in FROZEN_MODELS.items():

    partition = training_partitions[
        entry['training_dataset_sha256']
    ]

    assert entry['training_samples'] == int(
        dataset_records.loc[partition, 'Rows']
    )

    assert entry['validation_dataset_sha256'] == (
        frozen_manifest['validation_dataset_sha256']
    )

print('Training and validation datasets for all 19 runs: Verified')


# Verify split and leakage removal records

preparation_tables = (
    NOTEBOOKS_DIR /
    '02_data_preparation/results/tables'
)

leakage_path = (
    preparation_tables / 'leakage_removal_summary.csv'
)

split_summary_path = (
    preparation_tables / 'train_validation_split_summary.csv'
)

leakage = pd.read_csv(
    leakage_path
).set_index('Metric')['Count']

assert leakage[
    'Remaining normalized train-test overlaps'
] == 0

assert (
    leakage['Original training comments'] -
    leakage['Overlapping training comments removed']
) == leakage['Final development pool']

assert (
    dataset_records.loc['train_full', 'Rows'] +
    dataset_records.loc['validation', 'Rows']
) == leakage['Final development pool']


# Independently recheck exact ID and normalized text separation

def load_partition_keys(partition):

    path = PROJECT_ROOT / dataset_records.loc[
        partition, 'Dataset Path'
    ]

    frame = pd.read_parquet(
        path,
        columns=['id', 'comment_text']
    )

    ids = set(frame['id'].astype(str))

    keys = set(
        frame['comment_text']
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    assert len(ids) == len(frame)

    return ids, keys


train_ids, train_keys = load_partition_keys('train_full')
val_ids, val_keys = load_partition_keys('validation')
test_ids_set, test_keys = load_partition_keys('test_official')

assert not train_ids.intersection(val_ids)
assert not train_ids.intersection(test_ids_set)
assert not val_ids.intersection(test_ids_set)

assert not train_keys.intersection(val_keys)
assert not train_keys.intersection(test_keys)
assert not val_keys.intersection(test_keys)

representative_ids, representative_keys = load_partition_keys(
    'train_representative'
)

enriched_ids, enriched_keys = load_partition_keys(
    'train_enriched'
)

assert representative_ids.issubset(train_ids)
assert representative_keys.issubset(train_keys)

assert enriched_ids.issubset(train_ids)
assert enriched_keys.issubset(train_keys)

print('Cross-partition ID overlap: Zero')
print('Cross-partition normalized text overlap: Zero')
print('Both training subsets: Verified as train-only')

del train_ids, train_keys, val_ids, val_keys
del test_ids_set, test_keys
del representative_ids, representative_keys
del enriched_ids, enriched_keys

gc.collect()


# Part B: Verify core model and audit requirements

classical_count = sum(
    entry['kind'] == 'classical'
    for entry in FROZEN_MODELS.values()
)

transformer_count = sum(
    entry['kind'] == 'transformer'
    for entry in FROZEN_MODELS.values()
)

assert classical_count == 12
assert transformer_count == 7

required_loss_runs = [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42',
    'T04_BCE_S42',
    'T05_WBCE_S42',
    'T06_FOCAL_S42'
]

assert set(required_loss_runs).issubset(FROZEN_MODELS)

assert all(
    FROZEN_MODELS[rid]['training_samples'] ==
    dataset_records.loc['train_representative', 'Rows']
    for rid in required_loss_runs
)

assert all(
    entry['score_type'] == 'sigmoid_probability'
    for entry in FROZEN_MODELS.values()
    if entry['kind'] == 'transformer'
)

assert frozen_manifest['selection_partition'] == 'validation'
assert frozen_manifest['thresholds_frozen']
assert not frozen_manifest['official_test_used_for_selection']

assert len(primary_per_label) == 6

assert fairness_completion['identity_terms'] == 30
assert fairness_completion['identity_categories'] == 5
assert fairness_completion['natural_metric_records'] == 324
assert fairness_completion['synthetic_comments'] == 744

assert explainability_completion['explained_cases'] == 20
assert explainability_completion['target_labels_covered'] == 6
assert explainability_completion['written_interpretations'] == 20
assert explainability_completion['faithfulness_evaluation_records'] == 460

assert error_completion['review_cases'] == 180
assert error_completion['ai_assisted_review_completed']

print('Core experimental requirements: Verified')


# Part C: Build requirement evidence matrix

requirement_rows = []

COMPLETE = 'Completed'
PENDING = 'Pending'
REVIEW = 'Completed with human review pending'
DEFERRED = 'Deferred - optional'


def add_requirement(
    requirement_id,
    category,
    requirement,
    status,
    implementation,
    evidence_paths,
    limitation=''
):

    relative_paths = []

    for path in evidence_paths:

        path = Path(path)

        assert path.is_file(), path

        relative_paths.append(
            str(path.relative_to(PROJECT_ROOT))
        )

    requirement_rows.append({

        'Requirement ID': requirement_id,
        'Category': category,
        'Requirement': requirement,
        'Status': status,
        'Implementation and Evidence': implementation,
        'Evidence Paths': ' | '.join(relative_paths),
        'Limitation or Outstanding Action': limitation
    })


# Eight core requirements from the master execution plan

add_requirement(
    'M01', 'Core',
    'Use the Jigsaw Toxic Comment Classification dataset',
    COMPLETE,
    'Six-label dataset with all valid official test records.',
    [
        dataset_manifest_path,
        leakage_path,
        split_summary_path
    ]
)

add_requirement(
    'M02', 'Core',
    'Use a justified training subsample',
    COMPLETE,
    (
        f"Representative training subset: "
        f"{dataset_records.loc['train_representative', 'Rows']}; "
        f"full training pool: "
        f"{dataset_records.loc['train_full', 'Rows']}."
    ),
    [
        dataset_manifest_path,
        TABLES_DIR / 'final_validation_results.csv'
    ],
    'Representative subset used for controlled comparisons.'
)

add_requirement(
    'M03', 'Core',
    'Address imbalance using class weights or focal loss',
    COMPLETE,
    'BCE, weighted BCE and focal loss compared on both transformers.',
    [
        REGISTRY_PATH,
        TABLES_DIR / 'final_matched_research_comparisons.csv'
    ],
    'Single-seed comparisons; weighting is not universally better.'
)

add_requirement(
    'M04', 'Core',
    'Fine-tune a transformer with six independent sigmoid outputs',
    COMPLETE,
    'DistilBERT and RoBERTa fine-tuned and evaluated.',
    [
        FROZEN_MANIFEST_PATH,
        TABLES_DIR / 'T01_final_comparison_index.csv'
    ]
)

add_requirement(
    'M05', 'Core',
    'Tune decision thresholds per label on validation',
    COMPLETE,
    'Six validation-optimized thresholds frozen before test evaluation.',
    [
        verify_file(primary_model_entry['threshold_file']),
        FROZEN_MANIFEST_PATH,
        TABLES_DIR / 'T03_final_thresholds_and_cost.csv'
    ]
)

add_requirement(
    'M06', 'Core',
    'Report per-category precision, recall and F1',
    COMPLETE,
    'Six-label official test performance with AP and ROC-AUC.',
    [
        TABLES_DIR / 'final_primary_per_label_metrics.csv',
        TABLES_DIR / 'final_primary_confidence_intervals.csv'
    ]
)

add_requirement(
    'M07', 'Core',
    'Provide LIME or SHAP explanations for 15-20 predictions',
    COMPLETE,
    (
        f"{explainability_completion['explained_cases']} LIME explanations, "
        f"{explainability_completion['feature_attribution_records']} "
        'signed word attributions and written interpretations.'
    ),
    [
        completion_paths['Explainability'],
        EXPLAINABILITY_RESULTS_DIR /
        'tables/explainability_key_findings.csv'
    ],
    'Written interpretations are AI-assisted; human review pending.'
)

add_requirement(
    'M08', 'Core',
    'Perform an explicit identity-term fairness audit',
    COMPLETE,
    (
        'Natural false-positive disparities for 30 identity terms '
        'and five categories; 744 controlled synthetic comments.'
    ),
    [
        completion_paths['Fairness audit'],
        FAIRNESS_RESULTS_DIR /
        'tables/fairness_key_findings.csv'
    ],
    'Identity terms are lexical proxies; no claim of demographic fairness.'
)

# Deployment is a separate mandatory delivery requirement

add_requirement(
    'M09', 'Core',
    'Prepare the deployment brief',
    PENDING,
    'Planned moderation decision-support workflow.',
    [
        TABLES_DIR / 'T03_final_thresholds_and_cost.csv',
        TABLES_DIR / 'final_moderation_workload.csv',
        RUN_CONFIGS_DIR / 'final_model_efficiency.json'
    ],
    'Write and verify the deployment brief in Cell 4.'
)


# Research extensions and quality gates

add_requirement(
    'R01', 'Research extension',
    'Compare classical machine learning baselines',
    COMPLETE,
    f'{classical_count} classical configurations evaluated.',
    [TABLES_DIR / 'T01_final_comparison_index.csv']
)

add_requirement(
    'R02', 'Research extension',
    'Compare DistilBERT and RoBERTa architectures',
    COMPLETE,
    f'{transformer_count} transformer configurations evaluated.',
    [TABLES_DIR / 'final_matched_research_comparisons.csv'],
    'Architecture comparisons depend on matched training protocols.'
)

add_requirement(
    'R03', 'Research extension',
    'Investigate training-data scaling',
    COMPLETE,
    'Full-data scaling evaluated for LR and focal-loss RoBERTa.',
    [TABLES_DIR / 'final_matched_research_comparisons.csv'],
    'Data volume and optimizer-update budget jointly differ.'
)

add_requirement(
    'R04', 'Research extension',
    'Quantify test metric uncertainty',
    COMPLETE,
    '1000 normalized-comment-group bootstrap repetitions.',
    [
        TABLES_DIR / 'final_primary_confidence_intervals.csv',
        TABLES_DIR / 'final_paired_model_comparisons.csv'
    ],
    'Intervals are conditional on fixed trained models.'
)

add_requirement(
    'R05', 'Research extension',
    'Perform systematic qualitative error analysis',
    REVIEW,
    '180 AI-assisted diagnostic case annotations.',
    [completion_paths['Error analysis']],
    'Independent human validation remains pending.'
)

add_requirement(
    'R06', 'Research extension',
    'Evaluate explanation faithfulness',
    COMPLETE,
    'Top-five deletion, 400 random deletions and feature retention.',
    [
        completion_paths['Explainability'],
        EXPLAINABILITY_RESULTS_DIR /
        'tables/T03_faithfulness_per_case.csv'
    ],
    'Perturbed text may be unnatural; cases are diagnostic.'
)

add_requirement(
    'R07', 'Research extension',
    'Complete independent human review of interpretations',
    PENDING,
    'Twenty case-specific interpretations already prepared.',
    [completion_paths['Explainability']],
    'Student or independent reviewer must validate interpretations.'
)

add_requirement(
    'R08', 'Research extension',
    'Complete independent synthetic human validation',
    PENDING,
    'Synthetic frame labels currently represent designed intent.',
    [completion_paths['Fairness audit']],
    'Independent annotation has not been performed.'
)

add_requirement(
    'R09', 'Research extension',
    'Complete additional scaling, seed and ablation experiments',
    DEFERRED,
    'Deferred experiments recorded in frozen manifest.',
    [FROZEN_MANIFEST_PATH, DECISION_LOG_PATH],
    'No post-freeze test-driven retraining permitted.'
)

add_requirement(
    'Q01', 'Quality safeguard',
    'Prevent development-test leakage',
    COMPLETE,
    'ID and normalized-text overlap checks independently repeated.',
    [dataset_manifest_path, leakage_path],
    'Exact normalized matching does not detect every semantic duplicate.'
)

add_requirement(
    'Q02', 'Quality safeguard',
    'Freeze models before official test evaluation',
    COMPLETE,
    'Model selection and threshold selection based on validation.',
    [
        FROZEN_MANIFEST_PATH,
        completion_paths['Final evaluation']
    ],
    'Benchmark was historically familiar to the project team.'
)

add_requirement(
    'Q03', 'Quality safeguard',
    'Maintain reproducible experimental evidence',
    COMPLETE,
    f'{len(verified_inventory)} previous artifacts verified.',
    [
        REGISTRY_PATH,
        DECISION_LOG_PATH,
        TABLES_DIR / 'research_stage_verification.csv'
    ]
)

add_requirement(
    'D01', 'Final delivery',
    'Prepare report and presentation evidence index',
    PENDING,
    'Verified numerical and visual evidence is available.',
    [TABLES_DIR / 'T01_final_comparison_index.csv'],
    'Cell 5.'
)

add_requirement(
    'D02', 'Final delivery',
    'Prepare final model manifest and reproducibility README',
    PENDING,
    'Frozen model dependencies already verified.',
    [FROZEN_MANIFEST_PATH],
    'Cells 4 and 5.'
)

add_requirement(
    'D03', 'Final delivery',
    'Complete final report, repository update and project backup',
    PENDING,
    'Final documentation in progress.',
    [consolidation_path],
    'Complete after Notebook 11 verification.'
)

requirements = pd.DataFrame(requirement_rows)

assert requirements['Requirement ID'].is_unique

requirement_matrix_path = (
    TABLES_DIR / 'T02_requirement_evidence_matrix.csv'
)

save_csv_atomic(
    requirement_matrix_path,
    requirements
)


# Part D: Document optional deferred experiments

deferred_experiments = frozen_manifest[
    'deferred_experiments'
]

assert isinstance(deferred_experiments, list)

deferred_rows = []

for experiment_id in deferred_experiments:

    deferred_rows.append({
        'Experiment ID': experiment_id,
        'Status': 'Deferred',
        'Reason': str(frozen_manifest['defer_reason']),
        'Source': str(
            FROZEN_MANIFEST_PATH.relative_to(PROJECT_ROOT)
        ),
        'Included In Final Results': False,
        'Post Freeze Rule': (
            'Any future experiment is exploratory unless '
            'evaluated using a fresh independent benchmark.'
        )
    })

deferred_table = pd.DataFrame(deferred_rows)

deferred_path = (
    TABLES_DIR / 'deferred_experiments.csv'
)

save_csv_atomic(
    deferred_path,
    deferred_table
)


# Part E: Consolidate important research limitations

limitations_data = [

    (
        'L01', 'Dataset domain',
        'English Wikipedia comments do not represent every online platform.',
        'Validate on platform-specific data before deployment.'
    ),

    (
        'L02', 'Annotation quality',
        'Subjective toxicity labels and conflicting annotations introduce noise.',
        'Retain disagreement and use contextual human review.'
    ),

    (
        'L03', 'Class imbalance',
        'Rare categories have limited positive examples.',
        'Report per-label AP, F1 and support alongside aggregate metrics.'
    ),

    (
        'L04', 'Training randomness',
        'Completed configurations use a single fixed training seed.',
        'Repeat training with additional seeds in future research.'
    ),

    (
        'L05', 'Scaling comparison',
        'Full-data training also changes the optimizer-update budget.',
        'Do not attribute all improvements solely to training-data size.'
    ),

    (
        'L06', 'Generalization',
        'Validation performance exceeds official test performance.',
        'Report both partitions and investigate distribution differences.'
    ),

    (
        'L07', 'Truncation',
        'The transformer input window is limited to 256 tokens.',
        'Review long comments and investigate longer-context models.'
    ),

    (
        'L08', 'Fairness',
        'Residual identity-term false-positive disparities remain.',
        'Use human review for identity-related and ambiguous predictions.'
    ),

    (
        'L09', 'Identity measurement',
        'Lexical mentions do not establish actual demographic membership.',
        'Avoid claims of demographic parity or absence of discrimination.'
    ),

    (
        'L10', 'Synthetic fairness',
        'Fixed templates have limited linguistic diversity.',
        'Obtain independent human annotations and broader templates.'
    ),

    (
        'L11', 'Explainability',
        'LIME provides local approximations rather than causal explanations.',
        'Report fidelity, deletion and retention evidence together.'
    ),

    (
        'L12', 'Human review',
        'Qualitative error and explanation interpretations are AI-assisted.',
        'Complete independent review before claiming human validation.'
    ),

    (
        'L13', 'Calibration',
        'Sigmoid scores have not been established as calibrated probabilities.',
        'Evaluate calibration using separate development evidence.'
    ),

    (
        'L14', 'Deployment efficiency',
        'Offline GPU throughput is not single-request service latency.',
        'Benchmark end-to-end latency and memory in the target environment.'
    ),

    (
        'L15', 'Threshold trade-off',
        'Reducing moderation workload may increase missed positive comments.',
        'Preserve frozen thresholds and disclose both error costs.'
    ),

    (
        'L16', 'Benchmark familiarity',
        'The official benchmark was historically familiar to the team.',
        'Treat it as held out for this pipeline, not an unfamiliar dataset.'
    ),

    (
        'L17', 'Statistical uncertainty',
        'Bootstrap intervals do not capture training-seed variability.',
        'Treat intervals as conditional and unadjusted for multiplicity.'
    ),

    (
        'L18', 'Deferred experiments',
        'Some optional experiments were skipped due to compute constraints.',
        'Document skipped runs and never report them as completed.'
    )
]

limitations = pd.DataFrame(
    limitations_data,
    columns=[
        'Limitation ID',
        'Area',
        'Limitation',
        'Recommended Future Action'
    ]
)

limitations_path = (
    TABLES_DIR / 'research_limitations.csv'
)

save_csv_atomic(
    limitations_path,
    limitations
)


# Part F: Final requirement status counts

requirement_summary = (
    requirements.groupby(
        ['Category', 'Status']
    )
    .size()
    .reset_index(name='Requirements')
)

requirement_summary_path = (
    TABLES_DIR / 'requirement_status_summary.csv'
)

save_csv_atomic(
    requirement_summary_path,
    requirement_summary
)

core_requirements = requirements[
    requirements['Category'] == 'Core'
]

assert len(core_requirements) == 9

assert core_requirements[
    'Status'
].eq(COMPLETE).sum() == 8

assert core_requirements.loc[
    core_requirements['Status'] == PENDING,
    'Requirement ID'
].tolist() == ['M09']


# Part G: Register requirement artifacts

requirement_inventory = []

for path in [
    requirement_matrix_path,
    deferred_path,
    limitations_path,
    requirement_summary_path
]:

    requirement_inventory.append({

        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': int(path.stat().st_size),
        'sha256': calculate_sha256(path)
    })

for record in requirement_inventory:
    verify_file(record)


# Verify unchanged frozen protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']


requirement_completion = {

    'status': 'Completed',

    'requirement_basis': (
        'NLP04_Final_Project_Execution_Plan.pdf, '
        'research design and quality gates'
    ),

    'original_university_brief_independently_reverified': False,

    'core_requirements': len(core_requirements),

    'core_requirements_completed': int(
        core_requirements['Status'].eq(COMPLETE).sum()
    ),

    'core_requirements_pending': int(
        core_requirements['Status'].eq(PENDING).sum()
    ),

    'pending_core_requirement_ids': ['M09'],

    'requirement_matrix_records': len(requirements),

    'documented_limitations': len(limitations),

    'deferred_experiments': deferred_experiments,

    'prepared_datasets_verified': 5,

    'cross_partition_id_overlap': 0,
    'cross_partition_normalized_overlap': 0,

    'independent_human_review_completed': False,

    'deployment_brief_completed': False,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'consolidation_completion_sha256': (
        calculate_sha256(consolidation_path)
    ),

    'artifact_inventory': requirement_inventory,

    'model_training_performed': False,
    'additional_inference_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(

    LOGS_DIR / 'requirement_verification_summary.json',

    requirement_completion
)


# Display requirement results

print('\nCore requirement evidence matrix')

display(
    core_requirements[[
        'Requirement ID',
        'Requirement',
        'Status',
        'Implementation and Evidence'
    ]]
)

print('\nRequirement status summary')

display(requirement_summary)

print('\nOptional deferred experiments')

display(deferred_table[['Experiment ID', 'Status']])

print('\nResearch limitations')

display(limitations[['Limitation ID', 'Area', 'Limitation']])

print('\nCell 3 summary')

print('Core requirements documented:', len(core_requirements))
print('Core requirements completed: 8')
print('Core requirement pending: Deployment brief')
print('Total requirement matrix records:', len(requirements))
print('Research limitations documented:', len(limitations))
print('Deferred experiments:', len(deferred_experiments))
print('Prepared datasets verified: 5')
print('Cross-partition ID overlap: 0')
print('Cross-partition normalized overlap: 0')
print('Independent human review: Pending')
print('Verified new artifacts:', len(requirement_inventory))
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 3 completed successfully.')

In [ ]:
# Cell 4: Deployment brief and final model manifest

from datetime import datetime, timezone

# Verify previous completion records

requirement_completion_path = (
    LOGS_DIR / 'requirement_verification_summary.json'
)

requirement_completion = read_json(
    requirement_completion_path
)

assert requirement_completion['status'] == 'Completed'
assert requirement_completion['core_requirements_completed'] == 8

assert requirement_completion[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    consolidation_path
) == requirement_completion[
    'consolidation_completion_sha256'
]

for completion in [
    consolidation_completion,
    requirement_completion
]:
    for record in completion['artifact_inventory']:
        verify_file(record)

print('Previous documentation artifacts: Verified')


# Part A: Restore verified numerical evidence

def load_verified_table(path, inventory):

    path = Path(path)

    matches = [
        record for record in inventory
        if record['path'] == str(
            path.relative_to(PROJECT_ROOT)
        )
    ]

    assert len(matches) == 1, path

    verify_file(matches[0])

    return pd.read_csv(
        path,
        float_precision='round_trip'
    )


consolidation_inventory = (
    consolidation_completion['artifact_inventory']
)

final_comparison = load_verified_table(

    TABLES_DIR / 'T01_final_comparison_index.csv',

    consolidation_inventory
)

primary_per_label = load_verified_table(

    TABLES_DIR / 'final_primary_per_label_metrics.csv',

    consolidation_inventory
)

moderation_workload = load_verified_table(

    TABLES_DIR / 'final_moderation_workload.csv',

    consolidation_inventory
)

primary_intervals = load_verified_table(

    TABLES_DIR / 'final_primary_confidence_intervals.csv',

    consolidation_inventory
)

primary_result = final_comparison[
    final_comparison['Run ID'] == PRIMARY_RUN_ID
].iloc[0]

primary_per_label = (
    primary_per_label
    .set_index('Label')
    .loc[LABELS]
    .reset_index()
)

assert len(primary_per_label) == 6

np.testing.assert_allclose(
    primary_per_label['Threshold'],
    thresholds,
    atol=1e-12,
    rtol=0
)

default_workload = moderation_workload[
    moderation_workload['Operating Point'] == 'Default'
].iloc[0]

optimized_workload = moderation_workload[
    moderation_workload['Operating Point'] ==
    'Validation optimized'
].iloc[0]


# Restore efficiency evidence

efficiency_path = (
    RUN_CONFIGS_DIR / 'final_model_efficiency.json'
)

efficiency_record = next(

    record for record in consolidation_inventory

    if record['path'] == str(
        efficiency_path.relative_to(PROJECT_ROOT)
    )
)

verify_file(efficiency_record)

efficiency_summary = read_json(efficiency_path)

assert efficiency_summary['run_id'] == PRIMARY_RUN_ID

assert not efficiency_summary[
    'single_request_latency_measured'
]

assert not efficiency_summary[
    'production_deployment_tested'
]

print('Performance, thresholds and efficiency: Verified')


# Part B: Restore fairness evidence

natural_fairness = load_verified_table(

    FAIRNESS_RESULTS_DIR /
    'tables/natural_identity_pooled_comparison.csv',

    fairness_completion['artifact_inventory']
)

assert len(natural_fairness) == 9

roberta_toxic_fairness = natural_fairness[

    natural_fairness['Model']
    .astype(str)
    .str.contains('roberta', case=False, regex=False)

    &

    natural_fairness['Outcome']
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('toxic')
]

assert len(roberta_toxic_fairness) == 1

fairness_row = roberta_toxic_fairness.iloc[0]

identity_fpr = float(
    fairness_row['Subgroup FPR (%)']
)

reference_fpr = float(
    fairness_row['Reference FPR (%)']
)

fairness_gap = float(
    fairness_row['FPR Gap (pp)']
)

np.testing.assert_allclose(

    identity_fpr - reference_fpr,

    fairness_gap,

    atol=1e-8,
    rtol=0
)

print('Natural identity fairness evidence: Verified')


# Part C: Verify final model and preprocessing configuration

assert primary_model_entry['labels'] == LABELS

assert primary_model_entry['score_type'] == (
    'sigmoid_probability'
)

assert primary_model_entry['comparison_operator'] == '>='

assert primary_model_entry['maximum_length'] == 256

assert primary_model_entry['padding_multiple'] == 8

assert primary_model_entry['inference_precision'] == (
    'fp16_cuda'
)

for record in primary_model_entry['model_files']:
    verify_file(record)

for key in [
    'config',
    'threshold_file',
    'checkpoint_manifest'
]:
    verify_file(primary_model_entry[key])

model_config = read_json(
    MODEL_PATH / 'config.json'
)

assert model_config['model_type'] == 'roberta'

assert len(model_config['id2label']) == 6

assert [
    model_config['id2label'][str(i)]
    for i in range(6)
] == LABELS

print('Final checkpoint and label mapping: Verified')


# Part D: Create final model manifest

final_model_manifest = {

    'project': (
        'NLP-04 Multi-Label Toxic Comment Classification '
        'with Explainability'
    ),

    'run_id': PRIMARY_RUN_ID,

    'architecture': primary_model_entry['model'],

    'base_checkpoint': primary_model_entry['checkpoint'],

    'base_checkpoint_revision': (
        primary_model_entry['revision']
    ),

    'model_path': primary_model_entry['model_path'],

    'model_state_sha256': (
        primary_model_entry['model_state_sha256']
    ),

    'model_files': primary_model_entry['model_files'],

    'training_config': primary_model_entry['config'],

    'checkpoint_manifest': (
        primary_model_entry['checkpoint_manifest']
    ),

    'threshold_file': primary_model_entry['threshold_file'],

    'training': {

        'loss_function': str(
            registry_index.loc[PRIMARY_RUN_ID, 'loss_function']
        ),

        'seed': primary_model_entry['seed'],

        'training_samples': (
            primary_model_entry['training_samples']
        ),

        'training_dataset_sha256': (
            primary_model_entry['training_dataset_sha256']
        ),

        'validation_dataset_sha256': (
            primary_model_entry['validation_dataset_sha256']
        )
    },

    'preprocessing': {

        'input_type': 'Raw English comment text',

        'tokenizer_source': (
            'Tokenizer stored with the frozen checkpoint'
        ),

        'lowercase_input': False,

        'remove_stopwords': False,

        'remove_punctuation': False,

        'truncation': True,

        'truncation_side': 'right',

        'maximum_length': (
            primary_model_entry['maximum_length']
        ),

        'padding': 'Dynamic batch padding',

        'padding_side': 'right',

        'padding_multiple': (
            primary_model_entry['padding_multiple']
        )
    },

    'inference': {

        'output_labels': LABELS,

        'output_activation': 'Independent sigmoid',

        'number_of_outputs': 6,

        'evaluation_mode': True,

        'gradient_computation': False,

        'benchmark_precision': (
            primary_model_entry['inference_precision']
        ),

        'comparison_operator': '>=',

        'thresholds': (
            primary_model_entry['optimized_thresholds']
        ),

        'threshold_selection_partition': 'Validation',

        'moderation_flag_rule': (
            'At least one label meets its frozen threshold'
        ),

        'scores_calibrated': False
    },

    'official_test_evidence': {

        'dataset_sha256': EXPECTED_TEST_HASH,

        'samples': len(test_df),

        'macro_ap': float(
            primary_result['Test Macro AP']
        ),

        'macro_roc_auc': float(
            primary_result['Test Macro ROC-AUC']
        ),

        'macro_f1': float(
            primary_result['Test Macro F1']
        ),

        'micro_f1': float(
            primary_result['Test Micro F1']
        )
    },

    'efficiency': efficiency_summary,

    'deployment_scope': (
        'Human-assisted content moderation decision support'
    ),

    'production_ready': False,

    'production_deployment_tested': False,

    'automatic_comment_removal_recommended': False,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'model_parameters_frozen': True,

    'thresholds_frozen': True,

    'model_training_performed': False,

    'additional_inference_performed': False
}

final_model_manifest_path = (
    RUN_CONFIGS_DIR / 'final_model_manifest.json'
)

save_json_atomic(
    final_model_manifest_path,
    final_model_manifest
)

print('Final model manifest: Saved')


# Part E: Generate the deployment brief

def markdown_table(frame):

    columns = frame.columns.tolist()

    lines = [

        '| ' + ' | '.join(columns) + ' |',

        '| ' + ' | '.join(
            ['---'] * len(columns)
        ) + ' |'
    ]

    for row in frame.itertuples(
        index=False,
        name=None
    ):

        lines.append(

            '| ' + ' | '.join(
                str(value) for value in row
            ) + ' |'
        )

    return '\n'.join(lines)


# Format label-specific deployment results

deployment_threshold_table = (
    primary_per_label[[
        'Label',
        'Threshold',
        'Precision',
        'Recall',
        'F1'
    ]].copy()
)

deployment_threshold_table['Threshold'] = (
    deployment_threshold_table['Threshold']
    .map(lambda x: f'{x:.6f}')
)

for column in [
    'Precision',
    'Recall',
    'F1'
]:

    deployment_threshold_table[column] = (
        deployment_threshold_table[column]
        .map(lambda x: f'{x:.4f}')
    )

threshold_markdown = markdown_table(
    deployment_threshold_table
)

# Restore Macro AP confidence interval

ap_interval = primary_intervals[
    primary_intervals['Metric'] == 'Macro AP'
].iloc[0]

flag_rate = float(
    optimized_workload['Flagged Comments Per 1000']
)

false_flag_rate = float(
    optimized_workload[
        'False Flagged Comments Per 1000'
    ]
)

miss_rate = float(
    optimized_workload[
        'Missed Positive Comments Per 1000'
    ]
)

default_flag_rate = float(
    default_workload['Flagged Comments Per 1000']
)

default_miss_rate = float(
    default_workload[
        'Missed Positive Comments Per 1000'
    ]
)

brief = f"""# NLP-04: Deployment Brief

## 1. Intended use and model

The proposed system assists human moderators in identifying potentially
toxic English-language comments. It is not an automatic removal or
punishment system.

The selected model is RoBERTa with focal loss
(`{PRIMARY_RUN_ID}`), fine-tuned on
{primary_model_entry['training_samples']:,} Jigsaw training comments.
Model selection used validation Macro Average Precision.

Raw text is preserved and tokenized using the frozen RoBERTa tokenizer,
with right truncation at {primary_model_entry['maximum_length']} tokens
and dynamic padding to a multiple of eight. Six independent sigmoid
scores are produced, allowing multiple toxicity labels per comment.

## 2. Verified performance and thresholds

Evaluation used {len(test_df):,} labelled official Jigsaw test comments.

- Macro AP: {primary_result['Test Macro AP']:.4f}
- Macro AP 95% bootstrap interval:
  [{ap_interval['Lower 2.5%']:.4f}, {ap_interval['Upper 97.5%']:.4f}]
- Macro ROC-AUC: {primary_result['Test Macro ROC-AUC']:.4f}
- Macro F1: {primary_result['Test Macro F1']:.4f}
- Micro F1: {primary_result['Test Micro F1']:.4f}

The following decision thresholds were optimized exclusively on
validation data and frozen before official test evaluation.

{threshold_markdown}

A label is flagged when its score is greater than or equal to its
threshold. The full-precision thresholds in the model manifest, rather
than the rounded values above, must be used for implementation.

## 3. Proposed moderation workflow

1. Validate incoming comment text and reject invalid input safely.
2. Tokenize using the frozen preprocessing configuration.
3. Generate six scores and apply the corresponding thresholds.
4. Route comments with at least one positive flag to human moderation.
5. Provide optional LIME explanations for diagnostic review.
6. Let moderators consider context, quotation, counterspeech and
   identity-related language before making decisions.
7. Record decisions securely and monitor errors, workload and drift.

Unflagged comments are not guaranteed harmless and should remain
eligible for user reports and other safety checks.

## 4. Expected benchmark moderation workload

Per 1,000 observed official test comments, the optimized thresholds
would flag approximately {flag_rate:.2f} unique comments, including
{false_flag_rate:.2f} comments annotated negative across all six labels.
Approximately {miss_rate:.2f} comments annotated positive for at least
one label would be missed entirely.

Default thresholds flag {default_flag_rate:.2f} comments and miss
{default_miss_rate:.2f} positive comments per 1,000.

Threshold optimization reduces workload but increases missed positive
comments at the comment level. These are benchmark observations,
not guaranteed production rates.

## 5. Computational requirements

The frozen model directory contains approximately
{efficiency_summary['model_size_mib']:.2f} MiB of model artifacts.

Offline batched GPU inference processed
{efficiency_summary['offline_inference_comments_per_second']:.2f}
comments per second with batch size
{efficiency_summary['batch_size']}.

Peak allocated GPU memory was
{efficiency_summary['peak_gpu_allocated_gib']:.3f} GiB during the
recorded benchmark.

Amortized inference time was
{efficiency_summary['amortized_inference_ms_per_comment']:.3f}
milliseconds per comment. This excludes unmeasured service overhead
and must not be interpreted as single-request API latency.

Production latency, total application memory, CPU serving performance
and concurrent-request capacity require separate benchmarking.

## 6. Fairness and explainability

On the natural test audit, the toxic-label false-positive rate was
{identity_fpr:.2f}% for comments containing monitored identity terms,
compared with {reference_fpr:.2f}% for comments without those terms.
The observed gap was {fairness_gap:.2f} percentage points.

The audit covered {fairness_completion['identity_terms']} identity terms
and {fairness_completion['synthetic_comments']} synthetic comments.
Residual identity sensitivity requires contextual human review.

LIME explanations were generated for
{explainability_completion['explained_cases']} diagnostic cases.
Mean local surrogate R2 was
{explainability_completion['mean_local_surrogate_r2']:.4f}.
Top-feature removal produced larger absolute score changes than
matched random removal in
{explainability_completion['top_deletion_outperforms_random_cases']}
of those cases.

LIME is diagnostic rather than causal. Written interpretations are
AI-assisted and independent human validation remains pending.

## 7. Limitations and deployment safeguards

The training domain is English Wikipedia comments. Annotation noise,
rare labels, truncation, identity sensitivity and distribution shift
limit generalization. Sigmoid scores are not established as calibrated
probabilities. Bootstrap intervals do not capture training-seed
variability.

Before operational use, validate on representative platform-specific
data, measure serving latency, review fairness and define moderator
escalation policies. Monitor flag rates, false positives, false
negatives and identity-term disparities. Restrict access to private
comments and maintain an appropriate data-retention policy.

Any future model or threshold changes require separate development
and fresh evaluation evidence. No production system has been deployed
or validated in this study.
"""

# Save deployment brief as Markdown

deployment_brief_path = (
    REPORTS_DIR / 'deployment_brief.md'
)

temp_path = deployment_brief_path.with_suffix('.tmp')

temp_path.write_text(
    brief,
    encoding='utf-8'
)

assert temp_path.read_text(
    encoding='utf-8'
) == brief

temp_path.replace(deployment_brief_path)

assert deployment_brief_path.stat().st_size > 0

print('Deployment brief: Saved')


# Part F: Register evidence sources

source_paths = [

    FROZEN_MANIFEST_PATH,

    TABLES_DIR / 'T01_final_comparison_index.csv',

    TABLES_DIR / 'final_primary_per_label_metrics.csv',

    TABLES_DIR / 'T03_final_thresholds_and_cost.csv',

    TABLES_DIR / 'final_moderation_workload.csv',

    TABLES_DIR / 'final_primary_confidence_intervals.csv',

    efficiency_path,

    FAIRNESS_RESULTS_DIR /
    'tables/natural_identity_pooled_comparison.csv',

    completion_paths['Fairness audit'],

    completion_paths['Explainability']
]

deployment_sources = []

for path in source_paths:

    assert path.is_file()

    deployment_sources.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'sha256': calculate_sha256(path),

        'size_bytes': int(path.stat().st_size)
    })

for record in deployment_sources:
    verify_file(record)

source_index_path = (
    RUN_CONFIGS_DIR / 'deployment_evidence_sources.json'
)

save_json_atomic(

    source_index_path,

    {
        'primary_run_id': PRIMARY_RUN_ID,
        'sources': deployment_sources
    }
)


# Part G: Verify final deployment artifacts

deployment_inventory = []

for path in [

    deployment_brief_path,

    final_model_manifest_path,

    source_index_path
]:

    deployment_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(path.stat().st_size),

        'sha256': calculate_sha256(path)
    })

for record in deployment_inventory:
    verify_file(record)

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']


# Record deployment brief completion

deployment_completion = {

    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,

    'deployment_brief_completed': True,

    'final_model_manifest_completed': True,

    'deployment_brief_path': str(
        deployment_brief_path.relative_to(PROJECT_ROOT)
    ),

    'deployment_brief_sha256': calculate_sha256(
        deployment_brief_path
    ),

    'final_model_manifest_path': str(
        final_model_manifest_path.relative_to(PROJECT_ROOT)
    ),

    'core_requirement_completed': 'M09',

    'core_requirements_completed': 9,

    'core_requirements_pending': 0,

    'verified_evidence_sources': len(deployment_sources),

    'numerical_results_derived_from_saved_artifacts': True,

    'human_assisted_moderation_proposed': True,

    'production_deployment_tested': False,

    'production_latency_measured': False,

    'independent_human_validation_completed': False,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'requirement_completion_sha256': calculate_sha256(
        requirement_completion_path
    ),

    'artifact_inventory': deployment_inventory,

    'model_training_performed': False,

    'additional_inference_performed': False,

    'thresholds_changed': False,

    'test_driven_selection_performed': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(

    LOGS_DIR / 'deployment_completion_summary.json',

    deployment_completion
)


# Part H: Display the generated deployment brief

from IPython.display import Markdown

display(Markdown(brief))

print('\nCell 4 summary')

print('Deployment brief: Saved and verified')
print('Final model manifest: Saved and verified')
print('Deployment evidence sources:', len(deployment_sources))
print('Verified deployment artifacts:', len(deployment_inventory))
print('Core requirements completed: 9/9')
print('Production deployment: Not performed')
print('Independent human validation: Pending')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 4 completed successfully.')

In [ ]:
# Cell 5: Report/PPT evidence and reproducibility documentation

import shutil
from PIL import Image
from datetime import datetime, timezone

# Restore verified documentation records

deployment_completion_path = (
    LOGS_DIR / 'deployment_completion_summary.json'
)

deployment_completion = read_json(
    deployment_completion_path
)

assert deployment_completion['status'] == 'Completed'
assert deployment_completion['core_requirements_completed'] == 9

for completion in [
    consolidation_completion,
    requirement_completion,
    deployment_completion
]:

    assert completion['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    for record in completion['artifact_inventory']:
        verify_file(record)

print('Previous documentation artifacts: Verified')


# Part A: Define report and presentation evidence

PRESENTATION_DIR = ASSETS_DIR / 'presentation'
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)

evidence_rows = []
copied_figures = []
documentation_notes = []


def register_evidence(
    section,
    description,
    path,
    partition,
    run_scope,
    presentation=False
):

    path = Path(path)

    assert path.is_file(), path
    assert path.stat().st_size > 0

    relative_path = str(
        path.relative_to(PROJECT_ROOT)
    )

    sha = calculate_sha256(path)

    # Compare with historical fingerprints where available

    if relative_path in verified_inventory:

        verify_file(
            verified_inventory[relative_path]
        )

        verification_scope = (
            'Historical completion inventory verified'
        )

    else:

        verification_scope = (
            'Current source fingerprint recorded'
        )

    evidence_id = f'A{len(evidence_rows) + 1:03d}'

    asset_type = path.suffix.lower().lstrip('.')

    selected_copy = ''

    if asset_type == 'png':

        with Image.open(path) as img:

            assert img.format == 'PNG'
            assert img.width > 0
            assert img.height > 0

            img.verify()

    if presentation:

        assert asset_type == 'png'

        destination = (
            PRESENTATION_DIR /
            f'{evidence_id}_{path.name}'
        )

        if destination.exists():

            assert calculate_sha256(destination) == sha

        else:

            shutil.copy2(
                path,
                destination
            )

        assert calculate_sha256(destination) == sha

        selected_copy = str(
            destination.relative_to(PROJECT_ROOT)
        )

        copied_figures.append(destination)

    evidence_rows.append({

        'Asset ID': evidence_id,

        'Report Section': section,

        'Description': description,

        'Asset Type': asset_type,

        'Run Scope': run_scope,

        'Evaluation Partition': partition,

        'Original Path': relative_path,

        'Original SHA256': sha,

        'Size Bytes': int(path.stat().st_size),

        'Verification Scope': verification_scope,

        'Selected for Presentation': presentation,

        'Presentation Copy': selected_copy

    })


# Part B: Dataset and preprocessing evidence

EDA_RESULTS = (
    NOTEBOOKS_DIR / '01_dataset_eda/results'
)

DATA_PREP_RESULTS = (
    NOTEBOOKS_DIR / '02_data_preparation/results'
)

register_evidence(
    'Dataset and methodology',
    'Training label imbalance',
    EDA_RESULTS / 'figures/label_distribution.png',
    'Raw training dataset',
    'Dataset audit',
    presentation=True
)

register_evidence(
    'Dataset and methodology',
    'Label co-occurrence',
    EDA_RESULTS /
    'figures/label_cooccurrence_counts.png',
    'Raw training dataset',
    'Dataset audit',
    presentation=True
)

register_evidence(
    'Dataset and methodology',
    'Representative versus full training distributions',
    DATA_PREP_RESULTS /
    'figures/subset_label_distribution_comparison.png',
    'Training partitions',
    'Dataset preparation',
    presentation=True
)

register_evidence(
    'Dataset and methodology',
    'Token lengths and truncation',
    DATA_PREP_RESULTS /
    'figures/sequence_length_truncation_comparison.png',
    'Prepared partitions',
    'Dataset preparation'
)

register_evidence(
    'Dataset and methodology',
    'Leakage removal evidence',
    DATA_PREP_RESULTS /
    'tables/leakage_removal_summary.csv',
    'Development and official test',
    'Dataset preparation'
)

register_evidence(
    'Dataset and methodology',
    'Actual training and validation split sizes',
    DATA_PREP_RESULTS /
    'tables/train_validation_split_summary.csv',
    'Development',
    'Dataset preparation'
)

register_evidence(
    'Dataset and methodology',
    'Prepared dataset manifest',
    dataset_manifest_path,
    'All prepared partitions',
    'Dataset preparation'
)


# Part C: Model development evidence

CLASSICAL_RESULTS = (
    NOTEBOOKS_DIR /
    '03_classical_ml/06_classical_ml_comparison/results'
)

DISTIL_RESULTS = (
    NOTEBOOKS_DIR / '04_distilbert/results'
)

ROBERTA_RESULTS = (
    NOTEBOOKS_DIR / '05_roberta/results'
)

OPTIMIZATION_RESULTS = (
    NOTEBOOKS_DIR / '06_model_optimization/results'
)

register_evidence(
    'Classical machine learning',
    'Classical algorithm comparison',
    CLASSICAL_RESULTS /
    'figures/classical_common_algorithm_comparison.png',
    'Validation',
    'Representative classical configurations',
    presentation=True
)

register_evidence(
    'Classical machine learning',
    'Controlled representation and balancing comparisons',
    CLASSICAL_RESULTS /
    'tables/classical_controlled_ablation_comparison.csv',
    'Validation',
    'Classical ablations'
)

register_evidence(
    'Transformer experiments',
    'DistilBERT loss-function comparison',
    DISTIL_RESULTS /
    'figures/distilbert_consolidated_comparison.png',
    'Validation',
    'T01, T02, T03'
)

register_evidence(
    'Transformer experiments',
    'DistilBERT and RoBERTa comparison',
    ROBERTA_RESULTS /
    'figures/transformer_consolidated_comparison.png',
    'Validation',
    'T01-T06',
    presentation=True
)

register_evidence(
    'Transformer experiments',
    'Controlled transformer ablations',
    ROBERTA_RESULTS /
    'tables/transformer_controlled_ablation.csv',
    'Validation',
    'T01-T06'
)

# Preserve epoch histories for all completed transformer runs

for rid in [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42',
    'T04_BCE_S42',
    'T05_WBCE_S42',
    'T06_FOCAL_S42',
    PRIMARY_RUN_ID
]:

    if rid.startswith(('T01', 'T02', 'T03')):
        base = DISTIL_RESULTS

    elif rid.startswith(('T04', 'T05', 'T06')):
        base = ROBERTA_RESULTS

    else:
        base = OPTIMIZATION_RESULTS

    register_evidence(
        'Transformer training',
        'Epoch-level training and validation history',
        base / rid / 'tables/epoch_history.csv',
        'Training and validation',
        rid
    )


# Part D: Optimization and final evaluation evidence

register_evidence(
    'Optimization and model selection',
    'Full-data scaling comparison',
    OPTIMIZATION_RESULTS /
    'figures/F02_scale_comparison.png',
    'Validation',
    'Representative and full-data runs',
    presentation=True
)

register_evidence(
    'Optimization and model selection',
    'Per-label threshold search',
    OPTIMIZATION_RESULTS /
    'figures/F03_F1_threshold_panels.png',
    'Validation',
    PRIMARY_RUN_ID
)

register_evidence(
    'Optimization and model selection',
    'Frozen model selection evidence',
    OPTIMIZATION_RESULTS /
    'tables/T01_model_selection.csv',
    'Validation',
    'All completed configurations'
)

evaluation_figures = [

    (
        'F01_test_model_comparison.png',
        'Official test model comparison',
        'All 19 configurations',
        True
    ),

    (
        'F02_final_label_precision_recall_F1.png',
        'Final per-label precision, recall and F1',
        PRIMARY_RUN_ID,
        True
    ),

    (
        'F03_final_binary_confusion_matrices.png',
        'Six label-specific confusion matrices',
        PRIMARY_RUN_ID,
        False
    ),

    (
        'F04_final_PR_curves.png',
        'Final model precision-recall curves',
        PRIMARY_RUN_ID,
        False
    ),

    (
        'F05_default_vs_tuned_test.png',
        'Default versus validation-tuned thresholds',
        'Frozen evaluation roster',
        True
    ),

    (
        'F06_paired_CI.png',
        'Paired bootstrap comparison intervals',
        'Frozen comparison models',
        False
    )
]

for filename, description, run_scope, selected in (
    evaluation_figures
):

    register_evidence(

        'Final performance',

        description,

        EVALUATION_RESULTS_DIR /
        'figures' / filename,

        'Official labelled test',

        run_scope,

        presentation=selected
    )

for filename, description in [

    (
        'T01_final_comparison_index.csv',
        'Complete validation-ranked comparison'
    ),

    (
        'final_primary_per_label_metrics.csv',
        'Final six-label classification metrics'
    ),

    (
        'final_primary_confidence_intervals.csv',
        'Final model bootstrap confidence intervals'
    ),

    (
        'final_paired_model_comparisons.csv',
        'Paired model comparison intervals'
    ),

    (
        'T03_final_thresholds_and_cost.csv',
        'Frozen thresholds and benchmark costs'
    ),

    (
        'final_moderation_workload.csv',
        'Unique comment-level moderation workload'
    )

]:

    register_evidence(
        'Final performance and deployment',
        description,
        TABLES_DIR / filename,
        (
            'Validation and official test, separately named'
            if filename == 'T01_final_comparison_index.csv'
            else 'Official labelled test'
        ),
        'Run IDs recorded in source table'
    )


# Part E: Error analysis, fairness and explainability

# Use only verified aggregate figures.
# Do not copy individual comments or raw-token plots.

audit_figure_specs = [

    (
        'Error analysis',
        ERROR_RESULTS_DIR,
        error_completion,
        4,
        {'F02'}
    ),

    (
        'Identity-term fairness',
        FAIRNESS_RESULTS_DIR,
        fairness_completion,
        6,
        {'F02', 'F04'}
    ),

    (
        'Explainability',
        EXPLAINABILITY_RESULTS_DIR,
        explainability_completion,
        2,
        {'F01', 'F02'}
    )
]

for section, directory, completion, expected, prefixes in (
    audit_figure_specs
):

    aggregate_directory = directory / 'figures'

    figures = sorted([

        PROJECT_ROOT / record['path']

        for record in completion['artifact_inventory']

        if (
            (PROJECT_ROOT / record['path']).parent ==
            aggregate_directory

            and

            record['path'].lower().endswith('.png')
        )
    ])

    # Error analysis has previously verified figure artifacts,
    # which may be recorded separately from its main inventory.

    if section == 'Error analysis' and len(figures) != expected:

        figures = sorted(
            aggregate_directory.glob('*.png')
        )

        documentation_notes.append(
            'Error analysis aggregate figure filenames '
            'were restored from the results directory; '
            'historical hashes checked where available.'
        )

    assert len(figures) == expected, (
        section, len(figures)
    )

    for path in figures:

        register_evidence(

            section,

            path.stem.replace('_', ' '),

            path,

            (
                'Training, official test and synthetic audit'
                if section == 'Identity-term fairness'
                else 'Official test diagnostics'
            ),

            (
                'Frozen audit models'
                if section == 'Identity-term fairness'
                else PRIMARY_RUN_ID
            ),

            presentation=any(
                path.name.startswith(prefix)
                for prefix in prefixes
            )
        )


# Register key audit results

for section, path in [

    (
        'Error analysis',
        completion_paths['Error analysis']
    ),

    (
        'Identity-term fairness',
        FAIRNESS_RESULTS_DIR /
        'tables/fairness_key_findings.csv'
    ),

    (
        'Explainability',
        EXPLAINABILITY_RESULTS_DIR /
        'tables/explainability_key_findings.csv'
    ),

    (
        'Explainability',
        EXPLAINABILITY_RESULTS_DIR /
        'tables/T03_faithfulness_per_case.csv'
    ),

    (
        'Explainability',
        EXPLAINABILITY_RESULTS_DIR /
        'tables/report_explanation_case_selection.csv'
    )

]:

    register_evidence(
        section,
        path.stem.replace('_', ' '),
        path,
        'Audit-specific partitions',
        'Run IDs documented in audit evidence'
    )


# Part F: Private individual explanation appendix index

# Do not copy these files into the presentation directory.

private_explain_dir = (
    EXPLAINABILITY_RESULTS_DIR / 'private_explanations'
)

report_example_ids = (
    explainability_completion['report_example_ids']
)

private_rows = []

for i in range(1, 21):

    case_id = f'E{i:02d}'

    figure_path = (
        private_explain_dir /
        'figures' /
        f'{case_id}_explanation.png'
    )

    html_path = (
        private_explain_dir /
        'html' /
        f'{case_id}_explanation.html'
    )

    for path in [figure_path, html_path]:

        key = str(path.relative_to(PROJECT_ROOT))

        verify_file(verified_inventory[key])

    private_rows.append({

        'Case ID': case_id,

        'Report Example': (
            case_id in report_example_ids
        ),

        'Private Figure Path': str(
            figure_path.relative_to(PROJECT_ROOT)
        ),

        'Figure SHA256': calculate_sha256(
            figure_path
        ),

        'Private HTML Path': str(
            html_path.relative_to(PROJECT_ROOT)
        ),

        'HTML SHA256': calculate_sha256(
            html_path
        ),

        'Human Review': 'Pending',

        'Publication Rule': (
            'Keep private; review and redact before '
            'including in a public report or presentation.'
        )
    })

private_appendix_index = pd.DataFrame(private_rows)

assert len(private_appendix_index) == 20

assert private_appendix_index[
    'Report Example'
].sum() == 6

PRIVATE_DOC_DIR = RESULTS_DIR / 'private_documentation'

PRIVATE_DOC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

private_index_path = (
    PRIVATE_DOC_DIR /
    'private_explanation_appendix_index.csv'
)

save_csv_atomic(
    private_index_path,
    private_appendix_index
)


# Part G: Correct deployment brief terminology

deployment_brief_path = PROJECT_ROOT / (
    deployment_completion['deployment_brief_path']
)

assert calculate_sha256(
    deployment_brief_path
) == deployment_completion[
    'deployment_brief_sha256'
]

original_brief = deployment_brief_path.read_text(
    encoding='utf-8'
)

weights_bytes = sum(

    record['size_bytes']

    for record in primary_model_entry['model_files']

    if record['path'].endswith('.safetensors')
)

weights_mib = weights_bytes / (1024 ** 2)

model_files_bytes = sum(

    record['size_bytes']

    for record in primary_model_entry['model_files']
)

model_files_mib = model_files_bytes / (1024 ** 2)

np.testing.assert_allclose(
    weights_mib,
    efficiency_summary['model_size_mib'],
    atol=1e-8,
    rtol=0
)

old_phrase = (
    'The frozen model directory contains approximately\n'
    f"{efficiency_summary['model_size_mib']:.2f} "
    'MiB of model artifacts.'
)

new_phrase = (

    f'The frozen model weights occupy '
    f'{weights_mib:.2f} MiB. '

    f'The four required model and tokenizer files '
    f'together occupy {model_files_mib:.2f} MiB, '

    'excluding the separate checkpoint manifest '
    'and application overhead.'
)

assert original_brief.count(old_phrase) == 1

final_brief = original_brief.replace(
    old_phrase,
    new_phrase
)

assert final_brief.replace(
    new_phrase,
    old_phrase
) == original_brief

final_deployment_brief_path = (
    REPORTS_DIR / 'deployment_brief_final.md'
)

final_deployment_brief_path.write_text(
    final_brief,
    encoding='utf-8'
)

assert final_deployment_brief_path.read_text(
    encoding='utf-8'
) == final_brief

documentation_notes.append(
    'Corrected deployment brief size terminology. '
    'Original brief preserved. No model or result changed.'
)

register_evidence(
    'Deployment',
    'Final corrected deployment brief',
    final_deployment_brief_path,
    'Official benchmark and proposed workflow',
    PRIMARY_RUN_ID
)


# Part H: Save report/PPT asset index

asset_index = pd.DataFrame(evidence_rows)

assert asset_index['Asset ID'].is_unique
assert asset_index['Original Path'].is_unique

assert not asset_index[
    'Original Path'
].str.contains('private_explanations').any()

assert len(copied_figures) == int(
    asset_index['Selected for Presentation'].sum()
)

asset_index_path = (
    TABLES_DIR / 'report_ppt_asset_index.csv'
)

save_csv_atomic(
    asset_index_path,
    asset_index
)

presentation_index = asset_index[
    asset_index['Selected for Presentation']
].copy()

presentation_index_path = (
    TABLES_DIR / 'presentation_asset_index.csv'
)

save_csv_atomic(
    presentation_index_path,
    presentation_index
)

print('Report and presentation asset indices: Saved')


# Part I: Create reproducibility README

notebook_directories = [

    '00_project_setup',
    '01_dataset_eda',
    '02_data_preparation',
    '03_classical_ml',
    '04_distilbert',
    '05_roberta',
    '06_model_optimization',
    '07_final_evaluation',
    '08_error_analysis',
    '09_fairness_audit',
    '10_explainability',
    '11_final_documentation'
]

for directory in notebook_directories:

    assert (
        NOTEBOOKS_DIR / directory
    ).is_dir(), directory

execution_order = '\n'.join(

    f'{i + 1}. `{directory}/`'

    for i, directory in enumerate(
        notebook_directories
    )
)

threshold_lines = '\n'.join(

    f'- `{label}`: '
    f"{primary_model_entry['optimized_thresholds'][label]:.17g}"

    for label in LABELS
)

deferred_text = ', '.join(
    frozen_manifest['deferred_experiments']
)

readme = f"""# NLP-04 Reproducibility Guide

## Project

Multi-Label Toxic Comment Classification with Explainability.

The research compares classical TF-IDF classifiers, DistilBERT
and RoBERTa on a shared leakage-aware Jigsaw data protocol.

## Final selected model

- Run ID: `{PRIMARY_RUN_ID}`
- Base model: `{primary_model_entry['checkpoint']}`
- Base revision: `{primary_model_entry['revision']}`
- Training samples: {primary_model_entry['training_samples']:,}
- Training seed: {primary_model_entry['seed']}
- Maximum sequence length: {primary_model_entry['maximum_length']}
- Independent sigmoid outputs: {len(LABELS)}
- Selection metric: validation Macro Average Precision

## Execution order

Run the notebooks in the following order:

{execution_order}

The classical ML stage contains multiple submodules.

Use GPU for transformer training and GPU-dependent inference.
Dataset analysis, metric consolidation and final documentation
can be performed on CPU.

## Data requirements

Obtain the Jigsaw Toxic Comment Classification Challenge dataset:

https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data

Required source data include train.csv, test.csv and test_labels.csv.

Use the original setup and data preparation notebooks to verify
raw files and regenerate the prepared partitions.

The prepared dataset manifest is:

`01_data/splits/prepared_dataset_manifest.csv`

The official scored test partition contains {len(test_df):,}
comments with valid annotations for all six labels.

Training and validation data must remain separate from official test
data. Raw comments are preserved for modeling. Lowercase and
whitespace normalization are used only for duplicate detection.

## Environment and dependencies

Use the recorded experiment configurations and original notebook
environment setup rather than assuming the newest library versions
will reproduce identical results.

The GitHub publication includes observed dependency information
where available in requirements-observed.txt.

The public repository does not contain all datasets, model binaries,
training checkpoints or private evaluation artifacts.

Exact replay requires the original verified files or compatible
artifacts that pass the saved fingerprint checks.

## Frozen evaluation protocol

Do not tune thresholds or select models on the official test set.

The authoritative frozen manifest is:

`{FROZEN_MANIFEST_PATH.relative_to(PROJECT_ROOT)}`

SHA-256:

`{EXPECTED_FROZEN_HASH}`

The final checkpoint directory is:

`{primary_model_entry['model_path']}`

The model files and full-precision thresholds are documented in
results/configs/final_model_manifest.json.

## Frozen decision thresholds

{threshold_lines}

Apply each threshold independently using score >= threshold.

Do not replace these values with rounded report values or
re-optimize using the test labels.

## Verified official test results

- Macro AP: {primary_result['Test Macro AP']:.6f}
- Macro ROC-AUC: {primary_result['Test Macro ROC-AUC']:.6f}
- Macro F1: {primary_result['Test Macro F1']:.6f}
- Micro F1: {primary_result['Test Micro F1']:.6f}

All values are derived from saved evaluation artifacts.

## Research evidence

The final documentation directory contains:

- T01_final_comparison_index.csv
- T02_requirement_evidence_matrix.csv
- T03_final_thresholds_and_cost.csv
- report_ppt_asset_index.csv
- presentation_asset_index.csv
- research_limitations.csv
- deferred_experiments.csv
- deployment_brief_final.md
- final_model_manifest.json

The Cell 3 requirement matrix is a historical snapshot. Deployment
completion is recorded separately until the final verification cell.

Original figures and tables are the authoritative evidence.
Presentation copies preserve the same file contents and SHA-256 hashes.

## Explainability and fairness

The frozen final model has 20 LIME explanations and 460
faithfulness evaluation records.

The fairness audit includes natural identity-term comparisons
and controlled synthetic comments.

These analyses do not establish causal explanations, demographic
fairness or production safety.

Independent human validation remains pending for AI-assisted
qualitative interpretations and synthetic audit intent.

## Deferred experiments

The following experiments were deferred:

{deferred_text}

These are not included as completed runs.

Any new post-freeze model changes must be identified as exploratory
unless evaluated using a fresh independent benchmark.

## Publication and privacy

Do not publish raw comments, raw-token explanation HTML,
individual explanation figures containing comment words,
private interpretation worksheets, or per-comment identifiers
and prediction records without separate review.

Exclude private_explanations, private_documentation and other
private review directories from the public repository.

Review notebook outputs and source cells for embedded private
content before publication.

Aggregate tables and figures should be reviewed before sharing.

## Deployment scope

The model is intended to support human moderation decisions.

It has not been validated as a production service. Offline
GPU throughput is not single-request latency.

The final deployment brief documents workload, fairness,
thresholds, limitations and proposed monitoring.
"""

readme_path = (
    REPORTS_DIR / 'reproducibility_README.md'
)

readme_path.write_text(
    readme,
    encoding='utf-8'
)

assert readme_path.read_text(
    encoding='utf-8'
) == readme

print('Reproducibility README: Saved')


# Part J: Verify selected presentation copies

for row in presentation_index.to_dict('records'):

    source = PROJECT_ROOT / row['Original Path']

    destination = PROJECT_ROOT / row[
        'Presentation Copy'
    ]

    assert calculate_sha256(source) == (
        row['Original SHA256']
    )

    assert calculate_sha256(destination) == (
        row['Original SHA256']
    )

print('All selected presentation copies: Verified')


# Part K: Build final Cell 5 artifact inventory

documentation_inventory = []

for path in [

    asset_index_path,

    presentation_index_path,

    private_index_path,

    final_deployment_brief_path,

    readme_path,

    *copied_figures
]:

    documentation_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    })

for record in documentation_inventory:
    verify_file(record)

assert len(documentation_inventory) == (
    len(copied_figures) + 5
)

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    deployment_brief_path
) == deployment_completion[
    'deployment_brief_sha256'
]


# Record evidence preparation completion

evidence_completion = {

    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,

    'report_evidence_records': len(asset_index),

    'selected_presentation_figures': len(
        copied_figures
    ),

    'private_explanation_appendix_records': 20,

    'private_report_examples': 6,

    'reproducibility_readme_completed': True,

    'presentation_figures_copied_without_modification': True,

    'final_deployment_brief_path': str(
        final_deployment_brief_path.relative_to(
            PROJECT_ROOT
        )
    ),

    'final_deployment_brief_sha256': (
        calculate_sha256(
            final_deployment_brief_path
        )
    ),

    'original_deployment_brief_preserved': True,

    'model_weights_mib': weights_mib,

    'required_model_and_tokenizer_files_mib': (
        model_files_mib
    ),

    'documentation_notes': documentation_notes,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'deployment_completion_sha256': (
        calculate_sha256(
            deployment_completion_path
        )
    ),

    'artifact_inventory': documentation_inventory,

    'private_raw_comments_copied_to_presentation': False,

    'public_repository_updated': False,

    'model_training_performed': False,

    'additional_inference_performed': False,

    'thresholds_changed': False,

    'test_driven_selection_performed': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(

    LOGS_DIR / 'evidence_preparation_summary.json',

    evidence_completion
)


# Part L: Display results

print('\nReport evidence distribution')

display(

    asset_index.groupby(
        'Report Section'
    ).size().reset_index(
        name='Evidence Records'
    )
)

print('\nSelected presentation assets')

display(

    presentation_index[[
        'Asset ID',
        'Report Section',
        'Description'
    ]]
)

print('\nCell 5 summary')

print('Report evidence records:', len(asset_index))

print(
    'Selected presentation figures:',
    len(copied_figures)
)

print('Private explanation appendix records: 20')
print('Private report examples: 6')

print(
    'Model weights MiB:',
    round(weights_mib, 4)
)

print(
    'Required model and tokenizer files MiB:',
    round(model_files_mib, 4)
)

print('Final deployment brief: Saved')
print('Original deployment brief: Preserved')
print('Reproducibility README: Saved')

print(
    'Verified new artifacts:',
    len(documentation_inventory)
)

print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 5 completed successfully.')

In [ ]:
# Cell 6: Final project verification and completion

from datetime import datetime, timezone

# Part A: Restore all final documentation records

documentation_stage_paths = {

    'Results consolidation':
        LOGS_DIR / 'final_results_consolidation_summary.json',

    'Requirement verification':
        LOGS_DIR / 'requirement_verification_summary.json',

    'Deployment':
        LOGS_DIR / 'deployment_completion_summary.json',

    'Evidence preparation':
        LOGS_DIR / 'evidence_preparation_summary.json'
}

documentation_records = {

    stage: read_json(path)

    for stage, path in documentation_stage_paths.items()
}

for stage, record in documentation_records.items():

    assert record['status'] == 'Completed', stage

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    assert not record['model_training_performed']
    assert not record['additional_inference_performed']
    assert not record['thresholds_changed']
    assert not record['test_driven_selection_performed']

    print(stage, ': Verified')


# Restore final documentation records

consolidation_completion = documentation_records[
    'Results consolidation'
]

requirement_completion = documentation_records[
    'Requirement verification'
]

deployment_completion = documentation_records[
    'Deployment'
]

evidence_completion = documentation_records[
    'Evidence preparation'
]

# Verify the documentation completion chain

assert calculate_sha256(
    documentation_stage_paths['Results consolidation']
) == requirement_completion[
    'consolidation_completion_sha256'
]

assert calculate_sha256(
    documentation_stage_paths['Requirement verification']
) == deployment_completion[
    'requirement_completion_sha256'
]

assert calculate_sha256(
    documentation_stage_paths['Deployment']
) == evidence_completion[
    'deployment_completion_sha256'
]

print('Documentation completion chain: Verified')


# Part B: Verify all historical research artifacts

master_inventory = {}


def add_inventory_record(record):

    path = verify_file(record)

    key = str(path.relative_to(PROJECT_ROOT))

    if key in master_inventory:

        assert master_inventory[key] == record

    else:

        master_inventory[key] = record


def register_final_file(path):

    path = Path(path)

    assert path.is_file()
    assert path.stat().st_size > 0

    record = {

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    }

    add_inventory_record(record)


# Verify previous research stages

for stage, record in completion_records.items():

    for artifact in record['artifact_inventory']:

        add_inventory_record(artifact)

    print(
        stage,
        '| Verified artifacts:',
        len(record['artifact_inventory'])
    )

historical_artifact_count = len(master_inventory)

assert historical_artifact_count == 231


# Verify final documentation artifacts

for stage, record in documentation_records.items():

    for artifact in record['artifact_inventory']:

        add_inventory_record(artifact)

    print(
        stage,
        '| Verified artifacts:',
        len(record['artifact_inventory'])
    )

documentation_artifact_count = (
    len(master_inventory) -
    historical_artifact_count
)

assert documentation_artifact_count == 38

print('Historical research artifacts:', historical_artifact_count)
print('New documentation artifacts:', documentation_artifact_count)


# Part C: Verify the final research evidence index

asset_index_path = (
    TABLES_DIR / 'report_ppt_asset_index.csv'
)

asset_index = pd.read_csv(
    asset_index_path,
    keep_default_na=False
)

assert len(asset_index) == (
    evidence_completion['report_evidence_records']
)

assert asset_index['Asset ID'].is_unique
assert asset_index['Original Path'].is_unique

for row in asset_index.to_dict('records'):

    source = PROJECT_ROOT / row['Original Path']

    assert source.is_file()

    assert calculate_sha256(source) == (
        row['Original SHA256']
    )

    assert source.stat().st_size == row['Size Bytes']

    register_final_file(source)

    if row['Selected for Presentation']:

        destination = (
            PROJECT_ROOT / row['Presentation Copy']
        )

        assert calculate_sha256(destination) == (
            row['Original SHA256']
        )

        assert 'private_explanations' not in (
            row['Presentation Copy']
        )

        assert 'private_documentation' not in (
            row['Presentation Copy']
        )

        register_final_file(destination)

assert asset_index[
    'Selected for Presentation'
].sum() == 14

print('Report evidence records: 52 verified')
print('Presentation figures: 14 verified')


# Part D: Verify final model and frozen protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    TEST_PATH
) == EXPECTED_TEST_HASH

for record in primary_model_entry['model_files']:

    add_inventory_record(record)

for key in [
    'config',
    'threshold_file',
    'checkpoint_manifest'
]:

    add_inventory_record(
        primary_model_entry[key]
    )

final_model_manifest_path = (
    RUN_CONFIGS_DIR / 'final_model_manifest.json'
)

final_model_manifest = read_json(
    final_model_manifest_path
)

assert final_model_manifest['run_id'] == PRIMARY_RUN_ID

assert final_model_manifest[
    'model_state_sha256'
] == primary_model_entry[
    'model_state_sha256'
]

assert final_model_manifest[
    'inference'
]['thresholds'] == primary_model_entry[
    'optimized_thresholds'
]

assert final_model_manifest[
    'model_parameters_frozen'
]

assert final_model_manifest['thresholds_frozen']

assert not final_model_manifest[
    'production_deployment_tested'
]

assert not frozen_manifest[
    'official_test_used_for_selection'
]

print('Final model, tokenizer and thresholds: Verified')


# Part E: Final requirement evidence matrix

# Preserve the historical Cell 3 matrix

original_requirement_path = (
    TABLES_DIR / 'T02_requirement_evidence_matrix.csv'
)

final_requirements = pd.read_csv(
    original_requirement_path,
    keep_default_na=False
)

assert len(final_requirements) == 24

assert final_requirements[
    'Requirement ID'
].is_unique


def update_requirement(
    requirement_id,
    status,
    implementation,
    evidence_paths,
    limitation
):

    mask = final_requirements[
        'Requirement ID'
    ].eq(requirement_id)

    assert mask.sum() == 1

    paths = []

    for path in evidence_paths:

        path = Path(path)

        assert path.is_file()

        paths.append(
            str(path.relative_to(PROJECT_ROOT))
        )

    final_requirements.loc[
        mask, 'Status'
    ] = status

    final_requirements.loc[
        mask, 'Implementation and Evidence'
    ] = implementation

    final_requirements.loc[
        mask, 'Evidence Paths'
    ] = ' | '.join(paths)

    final_requirements.loc[
        mask, 'Limitation or Outstanding Action'
    ] = limitation


# Mark deployment brief as completed

final_deployment_brief_path = PROJECT_ROOT / (
    evidence_completion['final_deployment_brief_path']
)

assert calculate_sha256(
    final_deployment_brief_path
) == evidence_completion[
    'final_deployment_brief_sha256'
]

update_requirement(

    'M09',

    'Completed',

    'Deployment brief and frozen final model manifest completed.',

    [
        final_deployment_brief_path,
        final_model_manifest_path,
        LOGS_DIR / 'deployment_completion_summary.json'
    ],

    'Proposed decision-support system; production testing not performed.'
)


# Mark report and PPT evidence preparation as completed

update_requirement(

    'D01',

    'Completed',

    (
        f"{evidence_completion['report_evidence_records']} "
        'report evidence records and '
        f"{evidence_completion['selected_presentation_figures']} "
        'verified presentation figures.'
    ),

    [
        asset_index_path,
        TABLES_DIR / 'presentation_asset_index.csv'
    ],

    'Final report and PPT assembly remain pending.'
)


# Mark reproducibility documentation as completed

readme_path = (
    REPORTS_DIR / 'reproducibility_README.md'
)

update_requirement(

    'D02',

    'Completed',

    'Final model manifest and reproducibility README saved.',

    [
        final_model_manifest_path,
        readme_path,
        FROZEN_MANIFEST_PATH
    ],

    'Exact replay requires original external data and model files.'
)


# Keep final submission activities pending

update_requirement(

    'D03',

    'Pending',

    'All planned research notebooks and supporting evidence prepared.',

    [
        asset_index_path,
        readme_path,
        final_deployment_brief_path
    ],

    'Final report, PPT, GitHub update and complete project backup pending.'
)


final_core = final_requirements[
    final_requirements['Category'] == 'Core'
]

assert len(final_core) == 9

assert final_core[
    'Status'
].eq('Completed').all()

assert not requirement_completion[
    'original_university_brief_independently_reverified'
]

final_requirement_path = (
    TABLES_DIR /
    'T02_requirement_evidence_matrix_final.csv'
)

save_csv_atomic(
    final_requirement_path,
    final_requirements
)

final_requirement_status = (

    final_requirements.groupby(
        ['Category', 'Status']
    )

    .size()

    .reset_index(name='Requirements')
)

final_requirement_status_path = (
    TABLES_DIR /
    'final_requirement_status_summary.csv'
)

save_csv_atomic(
    final_requirement_status_path,
    final_requirement_status
)

print('Core requirements completed: 9/9')
print('Final requirement matrix: Saved')


# Part F: Restore verified final research metrics

final_comparison = pd.read_csv(

    TABLES_DIR / 'T01_final_comparison_index.csv',

    float_precision='round_trip'
)

primary_result = final_comparison[
    final_comparison['Run ID'] == PRIMARY_RUN_ID
].iloc[0]

assert len(final_comparison) == 19

assert final_comparison.iloc[0]['Run ID'] == (
    PRIMARY_RUN_ID
)

primary_per_label = pd.read_csv(

    TABLES_DIR / 'final_primary_per_label_metrics.csv',

    float_precision='round_trip'
)

moderation_workload = pd.read_csv(

    TABLES_DIR / 'final_moderation_workload.csv',

    float_precision='round_trip'
)

optimized_workload = moderation_workload[
    moderation_workload['Operating Point'] ==
    'Validation optimized'
].iloc[0]

assert len(primary_per_label) == 6

np.testing.assert_allclose(

    [
        primary_result['Test Macro AP'],
        primary_result['Test Macro ROC-AUC'],
        primary_result['Test Macro F1'],
        primary_result['Test Micro F1']
    ],

    [
        evaluation_completion['primary_test_macro_ap'],
        evaluation_completion['primary_test_macro_roc_auc'],
        evaluation_completion['primary_test_optimized_macro_f1'],
        evaluation_completion['primary_test_optimized_micro_f1']
    ],

    atol=1e-12,
    rtol=0
)


# Part G: Consolidate final project findings

final_research_summary = {

    'project': (
        'NLP-04 Multi-Label Toxic Comment Classification '
        'with Explainability'
    ),

    'final_model': PRIMARY_RUN_ID,

    'training_samples': int(
        primary_result['Training Samples']
    ),

    'evaluated_configurations': len(final_comparison),

    'official_test_samples': len(test_df),

    'validation_macro_ap': float(
        primary_result['Validation Macro AP']
    ),

    'validation_macro_f1': float(
        primary_result['Validation Optimized Macro F1']
    ),

    'test_macro_ap': float(
        primary_result['Test Macro AP']
    ),

    'test_macro_roc_auc': float(
        primary_result['Test Macro ROC-AUC']
    ),

    'test_macro_f1': float(
        primary_result['Test Macro F1']
    ),

    'test_micro_f1': float(
        primary_result['Test Micro F1']
    ),

    'test_macro_precision': float(
        primary_result['Test Macro Precision']
    ),

    'test_macro_recall': float(
        primary_result['Test Macro Recall']
    ),

    'test_hamming_loss': float(
        primary_result['Test Hamming Loss']
    ),

    'test_exact_match_accuracy': float(
        primary_result['Test Exact Match Accuracy']
    ),

    'flagged_comments_per_1000': float(
        optimized_workload[
            'Flagged Comments Per 1000'
        ]
    ),

    'false_flagged_comments_per_1000': float(
        optimized_workload[
            'False Flagged Comments Per 1000'
        ]
    ),

    'missed_positive_comments_per_1000': float(
        optimized_workload[
            'Missed Positive Comments Per 1000'
        ]
    ),

    'error_analysis_cases': error_completion[
        'review_cases'
    ],

    'identity_terms_audited': fairness_completion[
        'identity_terms'
    ],

    'synthetic_fairness_comments': fairness_completion[
        'synthetic_comments'
    ],

    'lime_explanations': explainability_completion[
        'explained_cases'
    ],

    'mean_lime_r2': explainability_completion[
        'mean_local_surrogate_r2'
    ],

    'faithfulness_evaluations': explainability_completion[
        'faithfulness_evaluation_records'
    ],

    'top_deletion_outperforms_random_cases':
        explainability_completion[
            'top_deletion_outperforms_random_cases'
        ],

    'retained_prediction_match_rate':
        explainability_completion[
            'retained_prediction_match_rate'
        ],

    'model_weights_mib': evidence_completion[
        'model_weights_mib'
    ],

    'required_model_and_tokenizer_files_mib':
        evidence_completion[
            'required_model_and_tokenizer_files_mib'
        ],

    'offline_inference_comments_per_second':
        efficiency_summary[
            'offline_inference_comments_per_second'
        ]
}

research_summary_path = (
    RUN_CONFIGS_DIR / 'final_research_summary.json'
)

save_json_atomic(
    research_summary_path,
    final_research_summary
)


# Part H: Create a concise final research overview

s = final_research_summary

research_overview = f"""# NLP-04 Final Research Overview

## Selected model

RoBERTa with focal loss: `{PRIMARY_RUN_ID}`.

The model was selected using validation Macro Average Precision.
It was trained on {s['training_samples']:,} comments.

A total of {s['evaluated_configurations']} model configurations
were evaluated using the same frozen official test protocol.

## Final performance

| Metric | Validation | Official test |
|---|---:|---:|
| Macro Average Precision | {s['validation_macro_ap']:.6f} | {s['test_macro_ap']:.6f} |
| Macro F1 | {s['validation_macro_f1']:.6f} | {s['test_macro_f1']:.6f} |
| Macro ROC-AUC | See validation evidence | {s['test_macro_roc_auc']:.6f} |
| Micro F1 | See validation evidence | {s['test_micro_f1']:.6f} |

Official labelled test population: {s['official_test_samples']:,}.

The selected model achieved the highest Macro AP among the
completed configurations. This does not establish superiority
over all existing approaches or state-of-the-art models.

## Explainability and fairness

- Error-analysis cases: {s['error_analysis_cases']}
- Identity terms audited: {s['identity_terms_audited']}
- Synthetic fairness comments: {s['synthetic_fairness_comments']}
- LIME explanations: {s['lime_explanations']}
- Mean local surrogate R2: {s['mean_lime_r2']:.6f}
- Faithfulness evaluations: {s['faithfulness_evaluations']}
- Top deletion exceeds random mean absolute change:
  {s['top_deletion_outperforms_random_cases']} cases
- Retained prediction agreement:
  {100 * s['retained_prediction_match_rate']:.2f}%

Residual identity-term sensitivity remains. LIME explanations
are local diagnostic approximations rather than causal evidence.

## Proposed moderation workload

Per 1,000 observed benchmark comments:

- Flagged comments: {s['flagged_comments_per_1000']:.2f}
- Incorrectly flagged all-label negative comments:
  {s['false_flagged_comments_per_1000']:.2f}
- Missed positive comments:
  {s['missed_positive_comments_per_1000']:.2f}

These are benchmark observations, not production forecasts.

The deployment proposal uses human-assisted moderation.
Production deployment has not been performed.

## Requirement completion

All nine core requirements documented in the agreed master
execution plan are completed.

The original university brief was not independently reverified
during the final documentation stage.

Independent human review of the AI-assisted qualitative
interpretations and synthetic fairness intent remains pending.

## Deferred experiments

{', '.join(frozen_manifest['deferred_experiments'])}

Deferred experiments are excluded from completed results.

## Remaining submission activities

- Independent human validation.
- Final academic report and PPT assembly.
- Public GitHub repository update with privacy-safe artifacts.
- Complete verified project backup.

## Main evidence

- T01_final_comparison_index.csv
- T02_requirement_evidence_matrix_final.csv
- T03_final_thresholds_and_cost.csv
- report_ppt_asset_index.csv
- research_limitations.csv
- deployment_brief_final.md
- reproducibility_README.md
- final_model_manifest.json

All reported numerical results are derived from verified saved
experimental evidence.
"""

research_overview_path = (
    REPORTS_DIR / 'final_research_overview.md'
)

research_overview_path.write_text(
    research_overview,
    encoding='utf-8'
)

assert research_overview_path.read_text(
    encoding='utf-8'
) == research_overview

print('Final research overview: Saved')


# Part I: Complete the final artifact inventory

additional_paths = [

    RUN_CONFIGS_DIR / 'final_documentation_setup.json',

    TABLES_DIR / 'research_stage_verification.csv',

    TABLES_DIR / 'verified_headline_results.csv',

    FROZEN_MANIFEST_PATH,
    REGISTRY_PATH,
    DECISION_LOG_PATH,
    dataset_manifest_path,

    *completion_paths.values(),

    *documentation_stage_paths.values(),

    final_requirement_path,
    final_requirement_status_path,

    research_summary_path,
    research_overview_path
]

for path in additional_paths:

    register_final_file(path)

final_inventory = sorted(

    master_inventory.values(),

    key=lambda record: record['path']
)

assert len(final_inventory) == len({

    record['path']

    for record in final_inventory
})

# Every registered file was verified during registration.

print('Consolidated final artifact inventory: Verified')


# Part J: Create the master project completion manifest

final_project_completion = {

    'project': (
        'NLP-04 Multi-Label Toxic Comment Classification '
        'with Explainability'
    ),

    'notebook': '11_final_documentation',

    'status': 'Completed',

    'project_status': (
        'Research and notebook documentation completed; '
        'human validation and final submission pending'
    ),

    'completed_notebook_stages': 12,

    'primary_run_id': PRIMARY_RUN_ID,

    'evaluated_configurations': len(FROZEN_MODELS),

    'core_requirements_completed': 9,

    'core_requirements_pending': 0,

    'requirement_basis': (
        requirement_completion['requirement_basis']
    ),

    'original_university_brief_independently_reverified': False,

    'official_test_samples': len(test_df),

    'cross_partition_id_overlap': 0,

    'cross_partition_normalized_overlap': 0,

    'research_summary': final_research_summary,

    'report_evidence_records': len(asset_index),

    'selected_presentation_figures': int(
        asset_index['Selected for Presentation'].sum()
    ),

    'historical_research_artifacts_verified':
        historical_artifact_count,

    'documentation_stage_artifacts_verified':
        documentation_artifact_count,

    'total_unique_inventory_artifacts':
        len(final_inventory),

    'artifact_inventory': final_inventory,

    'deployment_brief_completed': True,

    'final_model_manifest_completed': True,

    'reproducibility_documentation_completed': True,

    'report_ppt_evidence_index_completed': True,

    'final_academic_report_completed': False,

    'final_presentation_completed': False,

    'public_repository_update_completed': False,

    'final_project_backup_completed': False,

    'independent_human_validation_completed': False,

    'outstanding_human_review': {

        'error_analysis_cases': error_completion[
            'review_cases'
        ],

        'lime_interpretations': explainability_completion[
            'independent_human_review_pending_cases'
        ],

        'synthetic_fairness_validation_pending': not
            fairness_completion[
                'independent_synthetic_human_validation_completed'
            ]
    },

    'deferred_experiments': (
        frozen_manifest['deferred_experiments']
    ),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'test_dataset_sha256': EXPECTED_TEST_HASH,

    'stage_completion_hashes': {

        stage: calculate_sha256(path)

        for stage, path in {

            **completion_paths,

            **documentation_stage_paths

        }.items()
    },

    'final_requirement_matrix_path': str(
        final_requirement_path.relative_to(PROJECT_ROOT)
    ),

    'final_requirement_matrix_sha256':
        calculate_sha256(final_requirement_path),

    'model_parameters_frozen': True,

    'thresholds_frozen': True,

    'model_training_performed': False,

    'additional_inference_performed': False,

    'thresholds_changed': False,

    'test_driven_selection_performed': False,

    'production_deployment_tested': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

final_completion_path = (
    LOGS_DIR / 'final_project_completion_summary.json'
)

save_json_atomic(
    final_completion_path,
    final_project_completion
)


# Part K: Final frozen protocol verification

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert read_json(
    final_completion_path
) == final_project_completion

gc.collect()


# Part L: Display final research results

print('\nFinal core requirement status')

display(

    final_core[[
        'Requirement ID',
        'Requirement',
        'Status'
    ]]
)

print('\nFinal research summary')

display(

    pd.DataFrame([
        final_research_summary
    ]).T
)

print('\nFinal requirement status distribution')

display(final_requirement_status)

print('\nFINAL PROJECT VERIFICATION')

print('Notebook stages completed: 12/12')
print('Core requirements completed: 9/9')
print('Evaluated model configurations: 19')
print('Official test samples:', len(test_df))
print('Historical research artifacts:', historical_artifact_count)
print('Documentation artifacts:', documentation_artifact_count)
print('Total unique verified artifacts:', len(final_inventory))
print('Report evidence records:', len(asset_index))
print('Presentation figures: 14')
print('Deployment brief: Completed')
print('Reproducibility documentation: Completed')
print('Independent human validation: Pending')
print('Final report and PPT assembly: Pending')
print('GitHub update: Pending')
print('Final backup: Pending')
print('Model training and inference: None')
print('Frozen research protocol: Unchanged')

print('\nMaster completion manifest:', final_completion_path)

print('\nCell 6 completed successfully.')
print('Notebook 11 completed successfully.')
print('ALL PLANNED RESEARCH NOTEBOOKS COMPLETED.')